In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn tqdm
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.0 MB/s eta 0:00:00


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
Path to dataset files: /kaggle/input/customer-support-on-twitter


In [ ]:
import os

print("Dataset path:", path)

for root, dirs, files in os.walk(path):
    print("\nFolder:", root)
    for file in files:
        print("  ", file)

Dataset path: /kaggle/input/customer-support-on-twitter

Folder: /kaggle/input/customer-support-on-twitter
   sample.csv

Folder: /kaggle/input/customer-support-on-twitter/twcs
   twcs.csv


In [ ]:
import pandas as pd
import os

csv_path = os.path.join(path, "twcs", "twcs.csv")

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (2811774, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [ ]:
print(df["inbound"].value_counts())
print()
print(df["inbound"].value_counts(normalize=True))

inbound
True     1537843
False    1273931
Name: count, dtype: int64

inbound
True     0.54693
False    0.45307
Name: proportion, dtype: float64


In [ ]:
brand_counts = (
    df[df["inbound"] == False]["author_id"]
    .value_counts()
)

print("Top 30 brands/accounts:")
display(brand_counts.head(30).to_frame("reply_count"))

Top 30 brands/accounts:


,reply_count
author_id,
AmazonHelp,169840
AppleSupport,106860
Uber_Support,56270
SpotifyCares,43265
Delta,42253
Tesco,38573
AmericanAir,36764
TMobileHelp,34317
comcastcares,33031


In [ ]:
top_brands = brand_counts.head(25).index.tolist()

brand_stats = []

for brand in top_brands:

    brand_replies = df[
        (df["author_id"] == brand) &
        (df["inbound"] == False)
    ]

    brand_stats.append({
        "brand": brand,
        "brand_replies": len(brand_replies),
        "unique_customers_replied_to": brand_replies["in_response_to_tweet_id"].nunique()
    })

brand_stats_df = pd.DataFrame(brand_stats)

display(
    brand_stats_df.sort_values(
        "brand_replies",
        ascending=False
    )
)

,brand,brand_replies,unique_customers_replied_to
0,AmazonHelp,169840,155445
1,AppleSupport,106860,106696
2,Uber_Support,56270,55283
3,SpotifyCares,43265,41734
4,Delta,42253,36215
5,Tesco,38573,25315
6,AmericanAir,36764,36524
7,TMobileHelp,34317,33909
8,comcastcares,33031,30455
9,British_Airways,29361,24108


In [ ]:
# Create lookup of tweet_id -> row
tweet_lookup = df.set_index("tweet_id")

pairs = []

for _, row in df[df["inbound"] == False].iterrows():

    parent_id = row["in_response_to_tweet_id"]

    if pd.isna(parent_id):
        continue

    parent_id = int(parent_id)

    if parent_id not in tweet_lookup.index:
        continue

    parent = tweet_lookup.loc[parent_id]

    if parent["inbound"] == True:
        pairs.append({
            "brand": row["author_id"],
            "customer_tweet_id": parent_id,
            "customer_text": parent["text"],
            "brand_tweet_id": row["tweet_id"],
            "brand_response": row["text"]
        })

pairs_df = pd.DataFrame(pairs)

print("Total customer → brand pairs:", len(pairs_df))

display(pairs_df.head())

Total customer → brand pairs: 1261888


,brand,customer_tweet_id,customer_text,brand_tweet_id,brand_response
0,sprintcare,3,@sprintcare I have sent several private messag...,1,@115712 I understand. I would like to assist y...
1,sprintcare,5,@sprintcare I did.,4,@115712 Please send us a Private Message so th...
2,sprintcare,8,@sprintcare is the worst customer service,6,@115712 Can you please send us a private messa...
3,sprintcare,12,@sprintcare You gonna magically change your co...,11,@115713 This is saddening to hear. Please shoo...
4,sprintcare,16,@sprintcare Since I signed up with you....Sinc...,15,@115713 We understand your concerns and we'd l...


In [ ]:
AMAZON = "AmazonHelp"

amazon_pairs = pairs_df[
    pairs_df["brand"] == AMAZON
].copy()

print("AmazonHelp customer → brand pairs:", len(amazon_pairs))

display(
    amazon_pairs[
        ["customer_text", "brand_response"]
    ].sample(
        30,
        random_state=42
    )
)

AmazonHelp customer → brand pairs: 168814


,customer_text,brand_response
700561,@119959 @115828 I preordered this game 14 mont...,@520984 I'm sorry for the mix-up! Please conta...
263554,@AmazonHelp 2ème colis qui devait arriver aujo...,@242837 Que dit le suivi de votre colis s'il v...
326948,@115821 are y’all ever gonna transfer my $50 g...,"@315479 To confirm, has your account been clos..."
876290,@115821 dudes. You refunded me for the wrong i...,@622478 We greatly appreciate your honesty! Pl...
155564,@AmazonHelp Fuck u then why the hell u r havin...,@210361 Usually the products are delivered by ...
147347,@AmazonHelp I had a scheduled furniture assemb...,"@206017 Apologies for the delay, I'd like to l..."
943973,@115821 don’t tell me I got guaranteed deliver...,@229000 I'm sorry to hear about this! Have you...
178681,@AmazonHelp Amazon shipping.,@119036 Thanks for confirming with us! We'd li...
760443,@AmazonHelp just had chat with a representativ...,@555171 Thanks. Do keep us posted. ^HK
522258,@AmazonHelp My lights all appear here but my D...,"@421770 Hello, Alanna! We want to help! Contac..."


In [ ]:
amazon_pairs["customer_length"] = (
    amazon_pairs["customer_text"]
    .fillna("")
    .str.len()
)

print(
    amazon_pairs["customer_length"].describe()
)

count    168814.000000
mean        116.589033
std          59.101826
min           7.000000
25%          72.000000
50%         118.000000
75%         145.000000
max         365.000000
Name: customer_length, dtype: float64


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=30,
    max_features=100
)

X = vectorizer.fit_transform(
    amazon_pairs["customer_text"].fillna("")
)

words = vectorizer.get_feature_names_out()
counts = X.sum(axis=0).A1

word_df = pd.DataFrame({
    "phrase": words,
    "count": counts
}).sort_values(
    "count",
    ascending=False
)

display(word_df.head(50))

,phrase,count
10,amazonhelp,101660
43,https,30188
8,amazon,27556
3,115850,25603
1,115821,21580
58,order,21435
24,delivery,18517
67,prime,14270
23,delivered,10529
80,service,10468


In [14]:
import re
import html
import pandas as pd

amazon = pairs_df[pairs_df["brand"] == "AmazonHelp"].copy()

def clean_text(text):
    text = html.unescape(str(text))

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove Twitter mentions
    text = re.sub(r'@\w+', ' ', text)

    # Remove HTML entities
    text = re.sub(r'&\w+;', ' ', text)

    # Replace long numeric IDs/order numbers with token
    text = re.sub(r'\b\d{4,}\b', ' <NUM> ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

amazon["customer_clean"] = amazon["customer_text"].apply(clean_text)

amazon[["customer_text", "customer_clean"]].head(20)

,customer_text,customer_clean
80,amazonのfireTVstickが見れない😢,amazonのfireTVstickが見れない😢
81,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんで...
82,@AmazonHelp こちらこそありがとうございました。,こちらこそありがとうございました。
103,amazonプライムビデオ、再生エラーが多いです,amazonプライムビデオ、再生エラーが多いです
142,Way to drop the ball on customer service @1158...,Way to drop the ball on customer service so pi...
143,@AmazonHelp 3 different people have given 3 di...,3 different people have given 3 different answ...
144,@115823 I want my amazon payments account CLOS...,I want my amazon payments account CLOSED. dm m...
145,"@115825 also, beim Addams Family-Film in Prime...","also, beim Addams Family-Film in Prime sind Bi..."
146,"@AmazonHelp Okay, danke für die Info","Okay, danke für die Info"
147,@115828 How about you guys figure out my Xbox ...,How about you guys figure out my Xbox One X pr...


In [15]:
from collections import Counter
import re

counter = Counter()

for text in amazon["customer_clean"]:
    words = re.findall(r"\b[a-zA-Z]{2,}\b", text.lower())
    counter.update(words)

counter.most_common(50)

[('the', 69382),
 ('to', 67449),
 ('it', 50601),
 ('my', 44674),
 ('and', 42405),
 ('is', 38680),
 ('for', 33515),
 ('you', 32300),
 ('on', 27575),
 ('amazon', 27540),
 ('not', 27099),
 ('in', 25665),
 ('me', 25332),
 ('no', 23549),
 ('of', 23540),
 ('have', 22700),
 ('this', 22117),
 ('order', 21435),
 ('but', 19691),
 ('that', 19597),
 ('was', 18801),
 ('delivery', 18517),
 ('num', 18229),
 ('can', 15651),
 ('your', 15604),
 ('with', 15531),
 ('prime', 14269),
 ('from', 13730),
 ('de', 13252),
 ('be', 12825),
 ('now', 11580),
 ('they', 10882),
 ('what', 10727),
 ('delivered', 10529),
 ('service', 10466),
 ('so', 10385),
 ('get', 10250),
 ('do', 10168),
 ('been', 9997),
 ('just', 9694),
 ('an', 9685),
 ('customer', 9625),
 ('as', 9461),
 ('when', 9397),
 ('are', 9306),
 ('que', 9167),
 ('by', 9006),
 ('day', 8917),
 ('will', 8795),
 ('why', 8749)]

In [16]:
import re
import html

def clean_text(text):
    text = html.unescape(str(text))

    # URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Twitter mentions
    text = re.sub(r'@\w+', ' ', text)

    # HTML entities
    text = re.sub(r'&\w+;', ' ', text)

    # Long IDs/order numbers -> NUM
    text = re.sub(r'\b\d{4,}\b', ' NUM ', text)

    # Newlines
    text = re.sub(r'\s+', ' ', text).strip()

    return text

amazon["customer_clean"] = amazon["customer_text"].apply(clean_text)

In [17]:
print("AmazonHelp pairs:", len(amazon))
print("Unique customers:", amazon["customer_tweet_id"].nunique())

print("\nEmpty messages:")
print(amazon["customer_clean"].eq("").sum())

print("\nDuplicate customer messages:")
print(amazon["customer_clean"].duplicated().sum())

print("\nMessage length:")
print(amazon["customer_clean"].str.len().describe())

AmazonHelp pairs: 168814
Unique customers: 154976

Empty messages:
1425

Duplicate customer messages:
17480

Message length:
count    168814.000000
mean        100.064740
std          57.883357
min           0.000000
25%          57.000000
50%         100.000000
75%         128.000000
max         281.000000
Name: customer_clean, dtype: float64


In [18]:
for threshold in [5, 10, 20, 30]:
    count = (amazon["customer_clean"].str.len() < threshold).sum()
    print(f"< {threshold} characters:", count)

< 5 characters: 2506
< 10 characters: 4420
< 20 characters: 10519
< 30 characters: 18322


In [19]:
amazon_model = amazon[
    amazon["customer_clean"].notna() &
    (amazon["customer_clean"].str.strip() != "")
].copy()

print("Rows:", len(amazon_model))
print("Unique messages:", amazon_model["customer_clean"].nunique())
print("Duplicate rows:",
      amazon_model["customer_clean"].duplicated().sum())

Rows: 167389
Unique messages: 151333
Duplicate rows: 16056


In [20]:
label_sample = amazon_model.sample(
    n=50,
    random_state=42
)[
    ["customer_tweet_id", "customer_clean", "brand_response"]
].copy()

for i, row in enumerate(label_sample.itertuples(index=False), 1):
    print("\n" + "="*100)
    print(f"EXAMPLE {i}")
    print("CUSTOMER :", row.customer_clean)
    print("RESPONSE :", row.brand_response)


EXAMPLE 1
CUSTOMER : 特典などの布ポスターに始末糸や、ほつれがあった場合に交換の対象となるのでしょうか。
RESPONSE : @196708 ご質問ありがとうございます。Twitter上ではご注文状況を確認しご案内することができないため、大変恐れ入りますがカスタマーサービスへお問い合わせいただきますようお願いいたします。https://t.co/J6YEizo6qC SK

EXAMPLE 2
CUSTOMER : No it hasn’t. I just cancel one of the order that charge me $50. I order that copy that charge me $42 on Monday and I think the system glitched. So I can’t track the item
RESPONSE : @569108 I'm so sorry for the trouble. If you have a moment, please contact us by phone or chat here: https://t.co/hApLpMlfHN. ^LH

EXAMPLE 3
CUSTOMER : Still not received my product. I don't no what is the problem. It is out for delivery but still not received. 48 hours are complete.
RESPONSE : @213988 Kindly share your details here: https://t.co/beaaDm0muc and we'll have this checked. ^GK

EXAMPLE 4
CUSTOMER : my order no. 404- NUM - NUM . I don’t know why my prepaid order is still not deliver, I’m unable to reach to ur customercare
RESPONSE : @416810 Apologies. You may always reach our s

In [21]:
!pip -q install scikit-learn

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

cluster_sample = amazon_model.sample(
    n=min(20000, len(amazon_model)),
    random_state=42
)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=5
)

X = vectorizer.fit_transform(
    cluster_sample["customer_clean"]
)

print(X.shape)

(20000, 7166)


In [24]:
from sklearn.cluster import KMeans

# Number of exploratory clusters
k = 12

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

cluster_sample["cluster"] = kmeans.fit_predict(X)

print("Clustering completed.")
print("\nCluster sizes:")
print(
    cluster_sample["cluster"]
    .value_counts()
    .sort_index()
)

Clustering completed.

Cluster sizes:
cluster
0       758
1      1564
2       702
3     10046
4       867
5       976
6      1375
7       833
8       699
9       653
10      890
11      637
Name: count, dtype: int64


In [25]:
terms = vectorizer.get_feature_names_out()

for cluster_id in range(k):
    center = kmeans.cluster_centers_[cluster_id]
    top_indices = center.argsort()[-15:][::-1]

    print("\n" + "=" * 80)
    print(f"CLUSTER {cluster_id}")
    print("-" * 80)

    print(", ".join(terms[i] for i in top_indices))


CLUSTER 0
--------------------------------------------------------------------------------
delivered, today, order, product, delivered today, product delivered, delivery, ordered, says, order delivered, supposed delivered, supposed, parcel, yesterday, says delivered

CLUSTER 1
--------------------------------------------------------------------------------
amazon, amazon prime, prime, account, india, pay, amazon pay, order, amazon india, product, balance, amazon logistics, logistics, use, app

CLUSTER 2
--------------------------------------------------------------------------------
customer, service, customer service, care, customer care, worst, amazon, worst customer, order, pathetic, poor, contact, help, amazon customer, worst service

CLUSTER 3
--------------------------------------------------------------------------------
order, yes, thanks, prime, time, email, thank, days, today, don, did, account, ve, got, product

CLUSTER 4
----------------------------------------------------

In [26]:
for cluster_id in range(k):

    print("\n\n" + "#" * 100)
    print(f"CLUSTER {cluster_id}")
    print("#" * 100)

    examples = cluster_sample[
        cluster_sample["cluster"] == cluster_id
    ].sample(
        n=min(8, len(
            cluster_sample[cluster_sample["cluster"] == cluster_id]
        )),
        random_state=42
    )

    for i, row in enumerate(examples.itertuples(index=False), 1):
        print(f"\n{i}. {row.customer_clean}")



####################################################################################################
CLUSTER 0
####################################################################################################

1. dear service provider ,my product not delivered till now But your agents updated my product it as delivered Resolve this...

2. It said it would be delivered by 8pm today on the app

3. I ordered something on , it says it’s delivered.... but there’s nothing on my porch.... why. WHY?!?!

4. how many times i have to fill the form but there is no positive response from your side and i have called your customer representative several times but they make false commitment as i have attached screenshot finally when my product is delivered to me ???????????

5. amazon prime package was said to be delivered today by 9pm!!! Its almost 12am

6. I subscribed to Prime for better service. order was not delivered yet you sent me a delivered message. This is how Prime works?

7. That is 

In [27]:
INTENT_DEFINITIONS = {
    "delivery_not_received":
        "Order is marked delivered, but customer says they did not receive it.",

    "delivery_delayed":
        "Expected delivery date/time has passed or shipment is late.",

    "delivery_failed_or_wrong":
        "Failed delivery attempt, wrong address, wrong package, delivery driver issue, or incorrect delivery.",

    "order_status":
        "Customer asks about order status, dispatch, tracking, cancellation, or an order that disappeared.",

    "return_or_refund":
        "Customer wants a return, replacement related to return, or is waiting for a refund.",

    "payment_or_billing":
        "Payment, charge, duplicate charge, Amazon Pay balance, or billing problem.",

    "account_access":
        "Account login, password, registered phone number, or account access problem.",

    "prime_subscription":
        "Prime membership, Prime subscription, Prime benefits, or Prime-related charges.",

    "prime_video":
        "Prime Video playback, streaming, video access, or Prime Video subscription issue.",

    "product_or_device":
        "Amazon device/product is broken, malfunctioning, or technically not working.",

    "product_information":
        "Question about product availability, features, compatibility, offers, or purchasing information.",

    "customer_service_complaint":
        "Complaint about Amazon customer service, support agents, or unresolved support experience.",

    "fraud_or_security":
        "Phishing, suspicious Amazon messages, account security, or suspected fraud.",

    "other":
        "Message does not clearly belong to another intent."
}

In [28]:
import pandas as pd

# Remove empty messages
candidate_df = amazon_model[
    amazon_model["customer_clean"].str.strip().ne("")
].copy()

# Remove exact duplicate text only for candidate selection
candidate_unique = candidate_df.drop_duplicates(
    subset=["customer_clean"]
).copy()

print("Candidate unique messages:", len(candidate_unique))

label_candidates = candidate_unique.sample(
    n=min(1000, len(candidate_unique)),
    random_state=42
).copy()

label_candidates[
    ["customer_tweet_id", "customer_clean", "brand_response"]
].to_csv(
    "amazon_label_candidates.csv",
    index=False
)

print("Saved:", len(label_candidates))

Candidate unique messages: 151333
Saved: 1000


In [29]:
cluster_label_sample = (
    cluster_sample
    .groupby("cluster", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(100, len(x)),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("Cluster-balanced candidates:",
      len(cluster_label_sample))

Cluster-balanced candidates: 1200


/tmp/ipykernel_924/4032852289.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [30]:
cluster_label_sample[
    ["customer_tweet_id", "customer_clean", "brand_response", "cluster"]
].to_csv(
    "amazon_cluster_label_candidates.csv",
    index=False
)

In [31]:
golden_df = (
    cluster_label_sample
    .groupby("cluster", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(17, len(x)),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("Golden candidates:", len(golden_df))
print("\nCluster distribution:")
print(golden_df["cluster"].value_counts().sort_index())

Golden candidates: 204

Cluster distribution:
cluster
0     17
1     17
2     17
3     17
4     17
5     17
6     17
7     17
8     17
9     17
10    17
11    17
Name: count, dtype: int64


/tmp/ipykernel_924/3829618955.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [32]:
golden_df = golden_df.sample(
    n=200,
    random_state=42
).reset_index(drop=True)

print("Golden set size:", len(golden_df))

Golden set size: 200


In [33]:
golden_df["intent"] = ""

golden_df = golden_df[
    [
        "customer_tweet_id",
        "customer_clean",
        "brand_response",
        "cluster",
        "intent"
    ]
]

golden_df.to_csv(
    "amazon_golden_200_to_label.csv",
    index=False
)

display(golden_df.head(10))

,customer_tweet_id,customer_clean,brand_response,cluster,intent
0,2875083,amazon prime package was said to be delivered ...,@798163 I'm sorry your package hasn't arrived ...,0,
1,2075342,"dear service provider ,my product not delivere...","@613762 Please don't provide your details, we ...",0,
2,1377425,Item ordered on 8th Oct not yet received. Deli...,@440653 form: https://t.co/beaaDm0muc and I’ll...,6,
3,1274967,Y a-t-il un moyen d'avoir une adresse de factu...,"@418446 Bonjour, cela est possible. Sur la pag...",4,
4,1303674,Queria saber se há previsão para a página de p...,"@424634 Oi Anderson, desde quando apresenta o ...",3,
5,991845,I ordered a phone that was faulty and to retur...,@354946 Apologies for the ordeal. Please share...,2,
6,318090,chose to return an item Order# 403- NUM - NUM ...,@191900 Please don't provide your order detail...,7,
7,419839,need help in updating a shipping address. Orde...,@210354 from the link here: https://t.co/TxK11...,9,
8,420016,today I got SMS saying package delivered but I...,@215137 Kindly reach out to our support team h...,11,
9,2791945,Gracias 🙌🏻,"@734119 ¡Hola, Aldo! Nos alegra saber que reci...",10,


In [34]:
golden_eval = golden_df.copy()

print("Golden evaluation set:", len(golden_eval))

Golden evaluation set: 200


In [35]:
golden_eval.to_csv(
    "amazon_golden_eval_200.csv",
    index=False
)

In [36]:
INTENT_DEFINITIONS = {
    "delivery_not_received":
        "Tracking says delivered, but customer did not receive the package.",

    "delivery_delayed":
        "Package/order is late or expected delivery date has passed.",

    "delivery_failed_or_wrong":
        "Failed delivery attempt, wrong address, wrong package, damaged delivery, or delivery-driver problem.",

    "order_status":
        "Customer asks about order status, dispatch, tracking, cancellation, or missing/disappeared order.",

    "return_or_refund":
        "Customer wants a return/replacement, return pickup, or refund/status of a refund.",

    "payment_or_billing":
        "Payment, card charge, duplicate charge, Amazon Pay balance, or billing problem.",

    "account_access":
        "Login, password, account access, registered phone number, or account settings.",

    "prime_subscription":
        "Prime membership, Prime subscription, Prime benefits, or Prime-related charges.",

    "prime_video":
        "Prime Video playback, streaming, video access, or Prime Video subscription issue.",

    "product_or_device":
        "Amazon device/product malfunction or technical problem.",

    "product_information":
        "Product availability, features, compatibility, offers, or purchasing information.",

    "customer_service_complaint":
        "Complaint about Amazon customer service, support agents, or unresolved support.",

    "fraud_or_security":
        "Phishing, suspicious Amazon messages, suspected fraud, or security issue.",

    "other":
        "Greeting, thanks, conversation follow-up, unclear issue, or anything not covered above."
}

In [37]:
def show_for_labeling(df):
    for i, row in df.iterrows():
        print("\n" + "=" * 100)
        print(f"INDEX: {i}")
        print(f"TEXT: {row['customer_clean']}")
        print(f"\nHistorical Amazon response:")
        print(row["brand_response"])

In [38]:
show_for_labeling(golden_eval.head(20))


INDEX: 0
TEXT: amazon prime package was said to be delivered today by 9pm!!! Its almost 12am

Historical Amazon response:
@798163 I'm sorry your package hasn't arrived yet. Can you tell us who the carrier is? If you're unsure you can check here: https://t.co/Y5jpI9gRhE. ^AY

INDEX: 1
TEXT: dear service provider ,my product not delivered till now But your agents updated my product it as delivered Resolve this...

Historical Amazon response:
@613762 Please don't provide your details, we consider it personal information. Our Twitter page is visible to public. ^HD

INDEX: 2
TEXT: Item ordered on 8th Oct not yet received. Delivery date was 20th Oct. A Prime customer

Historical Amazon response:
@440653 form: https://t.co/beaaDm0muc and I’ll contact you soon. 2/2 ^SC

INDEX: 3
TEXT: Y a-t-il un moyen d'avoir une adresse de facturation différente de l'adresse de livraison ? Je ne trouve pas de telle option.

Historical Amazon response:
@418446 Bonjour, cela est possible. Sur la page "vérific

In [39]:
print(golden_eval["intent"].value_counts())

intent
    200
Name: count, dtype: int64


In [40]:
labels_0_19 = [
    "delivery_delayed",
    "delivery_not_received",
    "delivery_delayed",
    "payment_or_billing",
    "product_information",
    "return_or_refund",
    "return_or_refund",
    "order_status",
    "delivery_not_received",
    "other",
    "payment_or_billing",
    "customer_service_complaint",
    "account_access",
    "customer_service_complaint",
    "delivery_delayed",
    "customer_service_complaint",
    "product_or_device",
    "other",
    "other",
    "delivery_delayed"
]

golden_eval.loc[:19, "intent"] = labels_0_19

print(golden_eval.loc[:19, ["customer_clean", "intent"]])

                                       customer_clean  \
0   amazon prime package was said to be delivered ...   
1   dear service provider ,my product not delivere...   
2   Item ordered on 8th Oct not yet received. Deli...   
3   Y a-t-il un moyen d'avoir une adresse de factu...   
4   Queria saber se há previsão para a página de p...   
5   I ordered a phone that was faulty and to retur...   
6   chose to return an item Order# 403- NUM - NUM ...   
7   need help in updating a shipping address. Orde...   
8   today I got SMS saying package delivered but I...   
9                                          Gracias 🙌🏻   
10  J'ai effectué une commande le 14 sur Amazon.es...   
11  Seriously - unless you ACTUALLY help me, don’t...   
12  Are you not AmazonHelp? If can't help me pass ...   
13  from the past 25 days the issue is escalated 1...   
14  I think you are not interested to shipped my p...   
15  I am more angry about the appalling service th...   
16  I hav bought redmi 4 from a

In [41]:
print(golden_eval["intent"].value_counts())

intent
                              180
delivery_delayed                4
other                           3
customer_service_complaint      3
delivery_not_received           2
payment_or_billing              2
return_or_refund                2
order_status                    1
product_information             1
account_access                  1
product_or_device               1
Name: count, dtype: int64


In [42]:
print(golden_eval["intent"].value_counts(dropna=False))

intent
                              180
delivery_delayed                4
other                           3
customer_service_complaint      3
delivery_not_received           2
payment_or_billing              2
return_or_refund                2
order_status                    1
product_information             1
account_access                  1
product_or_device               1
Name: count, dtype: int64


In [43]:
golden_eval[
    [
        "customer_tweet_id",
        "customer_clean",
        "brand_response",
        "intent"
    ]
].to_csv(
    "amazon_golden_200_labeling.csv",
    index=False
)

print("Saved amazon_golden_200_labeling.csv")

Saved amazon_golden_200_labeling.csv


In [44]:
from google.colab import files

files.download("amazon_golden_200_labeling.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
labels_50_79 = [
    "fraud_or_security",
    "return_or_refund",
    "return_or_refund",
    "delivery_failed_or_wrong",
    "product_or_device",
    "delivery_delayed",
    "other",
    "product_or_device",
    "return_or_refund",
    "other",
    "customer_service_complaint",
    "customer_service_complaint",
    "delivery_delayed",
    "payment_or_billing",
    "delivery_delayed",
    "customer_service_complaint",
    "delivery_delayed",
    "other",
    "return_or_refund",
    "delivery_not_received",
    "other",
    "product_information",
    "other",
    "account_access",
    "payment_or_billing",
    "delivery_delayed",
    "delivery_delayed",
    "prime_subscription",
    "return_or_refund",
    "product_information"
]

golden_eval.loc[50:79, "intent"] = labels_50_79

print(golden_eval.loc[50:79, ["customer_tweet_id", "intent"]])

    customer_tweet_id                      intent
50             514658           fraud_or_security
51            1072427            return_or_refund
52            1450062            return_or_refund
53             295265    delivery_failed_or_wrong
54            1030185           product_or_device
55            1577210            delivery_delayed
56             697951                       other
57            2881289           product_or_device
58            2972860            return_or_refund
59             442756                       other
60            1268247  customer_service_complaint
61             177385  customer_service_complaint
62            1745573            delivery_delayed
63             977710          payment_or_billing
64             789305            delivery_delayed
65            2266870  customer_service_complaint
66            2539661            delivery_delayed
67             983393                       other
68              20734            return_or_refund


In [46]:
labels_80_109 = [
    "customer_service_complaint",
    "delivery_failed_or_wrong",
    "order_status",
    "delivery_delayed",
    "prime_subscription",
    "return_or_refund",
    "other",
    "return_or_refund",
    "other",
    "delivery_delayed",
    "account_access",
    "prime_subscription",
    "customer_service_complaint",
    "return_or_refund",
    "delivery_delayed",
    "delivery_delayed",
    "delivery_not_received",
    "account_access",
    "delivery_failed_or_wrong",
    "return_or_refund",
    "delivery_failed_or_wrong",
    "delivery_not_received",
    "other",
    "order_status",
    "delivery_not_received",
    "fraud_or_security",
    "return_or_refund",
    "delivery_delayed",
    "product_or_device",
    "delivery_delayed"
]

golden_eval.loc[80:109, "intent"] = labels_80_109

golden_eval.to_csv("amazon_golden_200_labeling.csv", index=False)

print("Labeled:", golden_eval["intent"].notna().sum())

Labeled: 200


In [47]:
golden_eval.to_csv("amazon_golden_200_labeling.csv", index=False)

from google.colab import files
files.download("amazon_golden_200_labeling.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
print(df.shape)
print(df.columns.tolist())

(2811774, 7)
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [49]:
amazon = df[df["author_id"] == "AmazonHelp"].copy()

print("AmazonHelp rows:", len(amazon))

AmazonHelp rows: 169840


In [50]:
amazon_authors = (
    df[~df["inbound"]]["author_id"]
    .value_counts()
    .head(20)
)

print(amazon_authors)

author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
Name: count, dtype: int64


In [51]:
# Keep only customer tweets that have a direct AmazonHelp response
customer_to_amazon = df[
    (df["inbound"] == True) &
    (df["response_tweet_id"].notna())
].copy()

# response_tweet_id can contain multiple IDs separated by commas.
# For now, keep rows where at least one response exists.
customer_to_amazon = customer_to_amazon[
    customer_to_amazon["response_tweet_id"].astype(str).str.strip() != ""
]

print("Customer tweets with responses:", len(customer_to_amazon))

Customer tweets with responses: 1303829


In [52]:
tweet_lookup = df.set_index("tweet_id")["text"].to_dict()

In [53]:
def get_first_response(response_ids):
    if pd.isna(response_ids):
        return None

    ids = str(response_ids).split(",")

    for tweet_id in ids:
        tweet_id = tweet_id.strip()

        if tweet_id in tweet_lookup:
            return tweet_lookup[tweet_id]

    return None


customer_to_amazon["brand_response"] = (
    customer_to_amazon["response_tweet_id"]
    .apply(get_first_response)
)

In [54]:
pairs = customer_to_amazon[
    customer_to_amazon["brand_response"].notna()
].copy()

pairs = pairs[
    (pairs["text"].notna()) &
    (pairs["text"].str.strip() != "") &
    (pairs["brand_response"].str.strip() != "")
]

print("Usable customer → response pairs:", len(pairs))

Usable customer → response pairs: 0


In [55]:
pairs[
    ["tweet_id", "text", "brand_response"]
].head(10)

,tweet_id,text,brand_response


In [56]:
# Normalize tweet IDs to strings
df["tweet_id_str"] = df["tweet_id"].astype(str).str.strip()

# Build lookup using normalized IDs
tweet_lookup = dict(
    zip(df["tweet_id_str"], df["text"])
)

# Function to get AmazonHelp responses
def get_responses(response_ids):
    if pd.isna(response_ids):
        return []

    results = []

    for tweet_id in str(response_ids).split(","):
        tweet_id = tweet_id.strip()

        if tweet_id in tweet_lookup:
            results.append(tweet_lookup[tweet_id])

    return results


# Customer tweets that have response IDs
customer_to_amazon = df[
    (df["inbound"] == True) &
    (df["response_tweet_id"].notna())
].copy()

# Extract responses
customer_to_amazon["brand_responses"] = (
    customer_to_amazon["response_tweet_id"]
    .apply(get_responses)
)

# Keep only rows where we found at least one response
pairs = customer_to_amazon[
    customer_to_amazon["brand_responses"].apply(len) > 0
].copy()

# Use the first response for our initial dataset
pairs["brand_response"] = pairs["brand_responses"].apply(lambda x: x[0])

# Remove empty messages
pairs = pairs[
    pairs["text"].notna() &
    pairs["brand_response"].notna()
]

print("Usable pairs:", len(pairs))

Usable pairs: 1303829


In [57]:
print(
    pairs[["text", "brand_response"]]
    .head(10)
    .to_string(index=False)
)

                                                                                                                                                                text                                                                                                                                       brand_response
                                                                                  @sprintcare I have sent several private messages and no one is responding as usual                            @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.
                                                                                                                                                  @sprintcare I did.                         @115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.
                                                          

In [58]:
# Normalize IDs
df["tweet_id_str"] = df["tweet_id"].astype(str).str.strip()

# Build lookup: tweet_id -> complete row
tweet_lookup = df.set_index("tweet_id_str").to_dict("index")

# Get AmazonHelp's tweet IDs
amazonhelp_tweet_ids = set(
    df.loc[
        (df["inbound"] == False) &
        (df["author_id"] == "AmazonHelp"),
        "tweet_id_str"
    ]
)

print("AmazonHelp tweet IDs:", len(amazonhelp_tweet_ids))

AmazonHelp tweet IDs: 169840


In [59]:
def get_amazonhelp_responses(response_ids):
    if pd.isna(response_ids):
        return []

    responses = []

    for tweet_id in str(response_ids).split(","):
        tweet_id = tweet_id.strip()

        if tweet_id in amazonhelp_tweet_ids:
            response_text = tweet_lookup[tweet_id]["text"]

            if pd.notna(response_text) and str(response_text).strip():
                responses.append(str(response_text).strip())

    return responses


# Customer tweets
customer_to_amazon = df[
    (df["inbound"] == True) &
    (df["response_tweet_id"].notna())
].copy()

# Find only AmazonHelp responses
customer_to_amazon["amazonhelp_responses"] = (
    customer_to_amazon["response_tweet_id"]
    .apply(get_amazonhelp_responses)
)

# Keep only customer → AmazonHelp conversations
amazon_pairs = customer_to_amazon[
    customer_to_amazon["amazonhelp_responses"].apply(len) > 0
].copy()

# First AmazonHelp response
amazon_pairs["brand_response"] = (
    amazon_pairs["amazonhelp_responses"]
    .apply(lambda x: x[0])
)

# Clean
amazon_pairs = amazon_pairs[
    amazon_pairs["text"].notna() &
    amazon_pairs["brand_response"].notna()
].copy()

print("AmazonHelp customer → response pairs:", len(amazon_pairs))

AmazonHelp customer → response pairs: 154976


In [60]:
def get_amazonhelp_responses(response_ids):
    if pd.isna(response_ids):
        return []

    responses = []

    for tweet_id in str(response_ids).split(","):
        tweet_id = tweet_id.strip()

        if tweet_id in amazonhelp_tweet_ids:
            response_text = tweet_lookup[tweet_id]["text"]

            if pd.notna(response_text) and str(response_text).strip():
                responses.append(str(response_text).strip())

    return responses


# Customer tweets
customer_to_amazon = df[
    (df["inbound"] == True) &
    (df["response_tweet_id"].notna())
].copy()

# Find only AmazonHelp responses
customer_to_amazon["amazonhelp_responses"] = (
    customer_to_amazon["response_tweet_id"]
    .apply(get_amazonhelp_responses)
)

# Keep only customer → AmazonHelp conversations
amazon_pairs = customer_to_amazon[
    customer_to_amazon["amazonhelp_responses"].apply(len) > 0
].copy()

# First AmazonHelp response
amazon_pairs["brand_response"] = (
    amazon_pairs["amazonhelp_responses"]
    .apply(lambda x: x[0])
)

# Clean
amazon_pairs = amazon_pairs[
    amazon_pairs["text"].notna() &
    amazon_pairs["brand_response"].notna()
].copy()

print("AmazonHelp customer → response pairs:", len(amazon_pairs))

AmazonHelp customer → response pairs: 154976


In [61]:
print(
    amazon_pairs[
        ["text", "brand_response"]
    ].head(10).to_string(index=False)
)

                                                                                                                                                text                                                                                                                                            brand_response
                                                                                    @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。                                                                                     @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
                                                                                                                       @AmazonHelp こちらこそありがとうございました。                                                                                                    @115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET
                                                                                           

In [62]:
# Create clean retrieval dataset
retrieval_df = amazon_pairs[
    ["tweet_id", "text", "brand_response"]
].copy()

# Rename columns
retrieval_df.columns = [
    "customer_tweet_id",
    "customer_text",
    "brand_response"
]

# Basic cleaning
retrieval_df["customer_text"] = (
    retrieval_df["customer_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

retrieval_df["brand_response"] = (
    retrieval_df["brand_response"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove extremely short/noisy examples
retrieval_df = retrieval_df[
    (retrieval_df["customer_text"].str.len() >= 10) &
    (retrieval_df["brand_response"].str.len() >= 10)
].copy()

# Remove exact duplicate customer/response pairs
retrieval_df = retrieval_df.drop_duplicates(
    subset=["customer_text", "brand_response"]
).reset_index(drop=True)

print("Retrieval examples:", len(retrieval_df))

Retrieval examples: 154910


In [63]:
retrieval_df.head(10)

,customer_tweet_id,customer_text,brand_response
0,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎて...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
1,274,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
2,272,amazonのfireTVstickが見れない😢,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
3,325,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,616,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
5,617,Way to drop the ball on customer service @1158...,@115820 I'm sorry we've let you down! Without ...
6,621,@115823 I want my amazon payments account CLOS...,@115822 I am unable to affect your account via...
7,623,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
8,624,"@115825 also, beim Addams Family-Film in Prime...","@115824 Hi, wir erhalten die Filme/Serien so v..."
9,627,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115827 Thanks for your patience. ^KM


In [64]:
retrieval_df.to_csv(
    "amazonhelp_retrieval_dataset.csv",
    index=False
)

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use a subset first for speed
retrieval_sample = retrieval_df.sample(
    n=min(50000, len(retrieval_df)),
    random_state=42
).reset_index(drop=True)

# TF-IDF representation
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000
)

X = vectorizer.fit_transform(
    retrieval_sample["customer_text"]
)

print("TF-IDF matrix:", X.shape)

TF-IDF matrix: (50000, 68640)


In [66]:
def retrieve_similar_cases(query, k=3):
    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(query_vec, X).ravel()

    top_indices = scores.argsort()[-k:][::-1]

    results = retrieval_sample.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]

    return results[
        ["customer_text", "brand_response", "similarity"]
    ]

In [67]:
query = "My package says delivered but I never received it."

results = retrieve_similar_cases(query, k=3)

print(results.to_string(index=False))

                                                                                 customer_text                                                                                                                                brand_response  similarity
@AmazonHelp This order says delivered but I never received it #403-2589731-9684305 Please help              @353288 Please don't provide your order details, we consider it personal information. Our Twitter page is visible to public. ^HA    0.587599
                                   @AmazonHelp my package says it was delivered and it wasn’t? @293134 I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze? ^KB    0.556059
                                      @AmazonHelp my package says delivered but isn't here.. 🙄                       @329205 I'm sorry, don't fret yet! Let's try these steps first: https://t.co/9zP49AX3hn. Let us know if this helps! ^KJ    0.547028


In [68]:
print("TF-IDF matrix:", X.shape)

TF-IDF matrix: (50000, 68640)


In [69]:
import pandas as pd

golden = pd.read_csv("amazon_golden_200_labeling.csv")

print(golden.shape)
print(golden["intent"].value_counts())
print("Missing labels:", golden["intent"].isna().sum())

(200, 5)
intent
delivery_delayed              16
return_or_refund              12
other                         11
customer_service_complaint     8
delivery_not_received          6
delivery_failed_or_wrong       4
payment_or_billing             4
account_access                 4
product_or_device              4
product_information            3
order_status                   3
prime_subscription             3
fraud_or_security              2
Name: count, dtype: int64
Missing labels: 120


In [71]:
labels_110_199 = [
    "delivery_failed_or_wrong",   # 110
    "other",                       # 111
    "delivery_delayed",            # 112
    "payment_or_billing",          # 113
    "customer_service_complaint",  # 114
    "order_status",                # 115
    "delivery_delayed",            # 116
    "order_status",                # 117
    "customer_service_complaint",  # 118
    "return_or_refund",            # 119
    "product_or_device",           # 120
    "delivery_delayed",            # 121
    "delivery_delayed",            # 122
    "order_status",                # 123
    "product_information",         # 124
    "delivery_not_received",       # 125
    "other",                       # 126
    "customer_service_complaint",  # 127
    "prime_subscription",          # 128
    "delivery_delayed",            # 129
    "return_or_refund",            # 130
    "other",                       # 131
    "other",                       # 132
    "return_or_refund",            # 133
    "customer_service_complaint",  # 134
    "delivery_failed_or_wrong",    # 135
    "product_information",         # 136
    "account_access",              # 137
    "delivery_delayed",            # 138
    "delivery_not_received",       # 139
    "delivery_delayed",            # 140
    "return_or_refund",            # 141
    "delivery_not_received",       # 142
    "other",                       # 143
    "delivery_delayed",            # 144
    "customer_service_complaint",  # 145
    "delivery_delayed",            # 146
    "product_or_device",           # 147
    "customer_service_complaint",  # 148
    "delivery_delayed",             # 149
    "order_status",                # 150
    "delivery_failed_or_wrong",    # 151
    "other",                       # 152
    "other",                       # 153
    "delivery_delayed",            # 154
    "delivery_delayed",            # 155
    "delivery_failed_or_wrong",    # 156
    "product_or_device",           # 157
    "order_status",                # 158
    "delivery_delayed",            # 159
    "delivery_delayed",            # 160
    "product_information",         # 161
    "delivery_failed_or_wrong",    # 162
    "delivery_not_received",       # 163
    "delivery_delayed",            # 164
    "other",                       # 165
    "delivery_failed_or_wrong",    # 166
    "delivery_failed_or_wrong",    # 167
    "order_status",                # 168
    "order_status",                # 169
    "customer_service_complaint",  # 170
    "delivery_not_received",       # 171
    "other",                       # 172
    "product_information",         # 173
    "customer_service_complaint",  # 174
    "order_status",                # 175
    "customer_service_complaint",  # 176
    "delivery_failed_or_wrong",    # 177
    "customer_service_complaint",  # 178
    "delivery_delayed",            # 179
    "customer_service_complaint",  # 180
    "customer_service_complaint",  # 181
    "customer_service_complaint",  # 182
    "delivery_delayed",            # 183
    "delivery_not_received",       # 184
    "other",                       # 185
    "customer_service_complaint",  # 186
    "payment_or_billing",          # 187
    "other",                       # 188
    "delivery_delayed",            # 189
    "product_information",         # 190
    "delivery_delayed",            # 191
    "customer_service_complaint",  # 192
    "other",                       # 193
    "delivery_not_received",       # 194
    "delivery_delayed",            # 195
    "account_access",              # 196
    "delivery_failed_or_wrong",    # 197
    "other",                       # 198
    "delivery_delayed"             # 199
]

golden_eval.loc[110:199, "intent"] = labels_110_199

# Verify correctly
golden_eval["intent"] = golden_eval["intent"].fillna("").astype(str).str.strip()

print("Total rows:", len(golden_eval))
print("Missing labels:", (golden_eval["intent"] == "").sum())
print("Labeled rows:", (golden_eval["intent"] != "").sum())

Total rows: 200
Missing labels: 30
Labeled rows: 170


In [72]:
print("Total rows:", len(golden_eval))
print("Missing labels:", (golden_eval["intent"] == "").sum())
print("Labeled rows:", (golden_eval["intent"] != "").sum())

Total rows: 200
Missing labels: 30
Labeled rows: 170


In [73]:
missing = golden_eval[
    golden_eval["intent"].fillna("").astype(str).str.strip() == ""
]

print("Missing rows:", len(missing))
print(missing[["customer_tweet_id", "customer_clean"]].to_string(index=False))

Missing rows: 30
 customer_tweet_id                                                                                                                                                                                                                                                                      customer_clean
           2473380                                                                                                                                         Ayer, martes 14 de noviembre, en el mail de confirmación salía entre el 11 y el 14, pero en la notificación de producto enviado ponía 14/11
            942729                                                                                                                                                                                                                                    - Really? This large box for just a SD card?....
            478554                                                                                

In [74]:
labels_20_49 = [
    "delivery_delayed",             # 20
    "other",                        # 21
    "customer_service_complaint",  # 22
    "payment_or_billing",           # 23
    "return_or_refund",             # 24
    "delivery_not_received",        # 25
    "other",                        # 26
    "return_or_refund",             # 27
    "delivery_failed_or_wrong",     # 28
    "other",                        # 29
    "account_access",               # 30
    "delivery_failed_or_wrong",     # 31
    "return_or_refund",             # 32
    "other",                        # 33
    "delivery_not_received",        # 34
    "other",                        # 35
    "delivery_not_received",        # 36
    "payment_or_billing",           # 37
    "return_or_refund",             # 38
    "delivery_not_received",        # 39
    "delivery_delayed",             # 40
    "other",                        # 41
    "delivery_delayed",             # 42
    "payment_or_billing",           # 43
    "product_information",          # 44
    "payment_or_billing",           # 45
    "delivery_failed_or_wrong",     # 46
    "prime_subscription",           # 47
    "delivery_delayed",             # 48
    "product_information"           # 49
]

golden_eval.loc[20:49, "intent"] = labels_20_49

# Clean and verify
golden_eval["intent"] = (
    golden_eval["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Total rows:", len(golden_eval))
print("Missing labels:", (golden_eval["intent"] == "").sum())
print("Labeled rows:", (golden_eval["intent"] != "").sum())

Total rows: 200
Missing labels: 0
Labeled rows: 200


In [75]:
golden_eval.to_csv(
    "amazon_golden_200_labeling.csv",
    index=False
)

from google.colab import files
files.download("amazon_golden_200_labeling.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [76]:
print(golden_eval["intent"].value_counts())

intent
delivery_delayed              41
other                         30
customer_service_complaint    24
return_or_refund              20
delivery_not_received         17
delivery_failed_or_wrong      16
order_status                  11
product_information           10
payment_or_billing            10
account_access                 7
product_or_device              7
prime_subscription             5
fraud_or_security              2
Name: count, dtype: int64


In [77]:
print("Unique intents:", golden_eval["intent"].nunique())
print(sorted(golden_eval["intent"].unique()))

Unique intents: 13
['account_access', 'customer_service_complaint', 'delivery_delayed', 'delivery_failed_or_wrong', 'delivery_not_received', 'fraud_or_security', 'order_status', 'other', 'payment_or_billing', 'prime_subscription', 'product_information', 'product_or_device', 'return_or_refund']


In [78]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Majority class from the training/discovery data
majority_intent = retrieval_df["intent"].mode()[0] if "intent" in retrieval_df.columns else "delivery_delayed"

# For this dataset, use the observed golden-set majority as the simple baseline
majority_intent = golden_eval["intent"].mode()[0]

y_true = golden_eval["intent"]
y_pred = [majority_intent] * len(y_true)

print("Majority intent:", majority_intent)
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))

print("\nClassification report:")
print(classification_report(y_true, y_pred, zero_division=0))

Majority intent: delivery_delayed
Accuracy: 0.205
Macro F1: 0.026172997127353975

Classification report:
                            precision    recall  f1-score   support

            account_access       0.00      0.00      0.00         7
customer_service_complaint       0.00      0.00      0.00        24
          delivery_delayed       0.20      1.00      0.34        41
  delivery_failed_or_wrong       0.00      0.00      0.00        16
     delivery_not_received       0.00      0.00      0.00        17
         fraud_or_security       0.00      0.00      0.00         2
              order_status       0.00      0.00      0.00        11
                     other       0.00      0.00      0.00        30
        payment_or_billing       0.00      0.00      0.00        10
        prime_subscription       0.00      0.00      0.00         5
       product_information       0.00      0.00      0.00        10
         product_or_device       0.00      0.00      0.00         7
          

In [79]:
print(retrieval_df.columns.tolist())
print(retrieval_df.shape)

['customer_tweet_id', 'customer_text', 'brand_response']
(154910, 3)


In [80]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

X_text = golden_eval["customer_clean"]
y = golden_eval["intent"]

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        max_features=30000,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

y_pred = cross_val_predict(
    model,
    X_text,
    y,
    cv=cv,
    method="predict"
)

print("TF-IDF + Logistic Regression")
print("Accuracy:", accuracy_score(y, y_pred))
print("Macro F1:", f1_score(y, y_pred, average="macro", zero_division=0))

print("\nClassification report:")
print(classification_report(y, y_pred, zero_division=0))

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


TF-IDF + Logistic Regression
Accuracy: 0.43
Macro F1: 0.3481567523710956

Classification report:
                            precision    recall  f1-score   support

            account_access       0.50      0.29      0.36         7
customer_service_complaint       0.46      0.54      0.50        24
          delivery_delayed       0.53      0.41      0.47        41
  delivery_failed_or_wrong       0.22      0.12      0.16        16
     delivery_not_received       0.32      0.65      0.43        17
         fraud_or_security       0.00      0.00      0.00         2
              order_status       0.67      0.36      0.47        11
                     other       0.45      0.50      0.48        30
        payment_or_billing       0.25      0.20      0.22        10
        prime_subscription       0.33      0.80      0.47         5
       product_information       0.50      0.10      0.17        10
         product_or_device       1.00      0.14      0.25         7
          return_o

In [81]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use the cleaned historical dataset
retrieval_sample = (
    retrieval_df
    .sample(n=min(50000, len(retrieval_df)), random_state=42)
    .reset_index(drop=True)
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

X = vectorizer.fit_transform(retrieval_sample["customer_text"])

print("Historical cases:", len(retrieval_sample))
print("TF-IDF matrix:", X.shape)

Historical cases: 50000
TF-IDF matrix: (50000, 68640)


In [82]:
def retrieve_similar_cases(query, k=3):
    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(query_vec, X).ravel()

    top_indices = scores.argsort()[-k:][::-1]

    results = retrieval_sample.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]

    return results[
        ["customer_text", "brand_response", "similarity"]
    ].reset_index(drop=True)

In [83]:
query = "My package says delivered but I never received it."

results = retrieve_similar_cases(query, k=5)

for i, row in results.iterrows():
    print("=" * 80)
    print(f"CASE {i+1}")
    print(f"Similarity: {row['similarity']:.3f}")
    print(f"Customer: {row['customer_text']}")
    print(f"AmazonHelp: {row['brand_response']}")

CASE 1
Similarity: 0.588
Customer: @AmazonHelp This order says delivered but I never received it #403-2589731-9684305 Please help
AmazonHelp: @353288 Please don't provide your order details, we consider it personal information. Our Twitter page is visible to public. ^HA
CASE 2
Similarity: 0.556
Customer: @AmazonHelp my package says it was delivered and it wasn’t?
AmazonHelp: @293134 I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze? ^KB
CASE 3
Similarity: 0.547
Customer: @AmazonHelp my package says delivered but isn't here.. 🙄
AmazonHelp: @329205 I'm sorry, don't fret yet! Let's try these steps first: https://t.co/9zP49AX3hn. Let us know if this helps! ^KJ
CASE 4
Similarity: 0.509
Customer: @AmazonHelp can u tell me why my package says delivered yet I haven’t received?
AmazonHelp: @194973 I'm sorry to hear you can't locate your parcel! Here's a link to our Help page for this situation: https://t.co/0b6AjQv76

In [84]:
import numpy as np

def retrieval_stats(eval_df, k=3):
    max_scores = []
    avg_scores = []

    for query in eval_df["customer_clean"]:
        query_vec = vectorizer.transform([query])
        scores = cosine_similarity(query_vec, X).ravel()

        top_scores = np.sort(scores)[-k:]

        max_scores.append(top_scores[-1])
        avg_scores.append(top_scores.mean())

    return {
        "mean_top1_similarity": np.mean(max_scores),
        "mean_top{}_similarity".format(k): np.mean(avg_scores),
        "median_top1_similarity": np.median(max_scores)
    }

stats = retrieval_stats(golden_eval, k=3)

print(stats)

{'mean_top1_similarity': np.float64(0.5511586902939891), 'mean_top3_similarity': np.float64(0.3913240788648048), 'median_top1_similarity': np.float64(0.42075254703215315)}


In [85]:
import re

def is_useful_historical_response(response):
    response = str(response).strip().lower()

    # Too short to contain a useful resolution
    if len(response) < 40:
        return False

    # Privacy / personal-information warnings
    privacy_patterns = [
        "don't provide your order details",
        "do not provide your order details",
        "personal information",
        "private information",
    ]

    if any(p in response for p in privacy_patterns):
        return False

    # Pure routing / acknowledgement responses
    generic_patterns = [
        "thanks for reaching out",
        "thank you for reaching out",
        "please contact us",
        "we're here to help",
    ]

    if any(p in response for p in generic_patterns) and len(response) < 100:
        return False

    return True

In [86]:
def retrieve_better_cases(query, k=10, final_k=3):
    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(query_vec, X).ravel()

    top_indices = scores.argsort()[-k:][::-1]

    candidates = retrieval_sample.iloc[top_indices].copy()
    candidates["similarity"] = scores[top_indices]

    candidates = candidates[
        candidates["brand_response"].apply(is_useful_historical_response)
    ]

    return (
        candidates
        .sort_values("similarity", ascending=False)
        .head(final_k)
        [["customer_text", "brand_response", "similarity"]]
        .reset_index(drop=True)
    )

In [87]:
query = "My package says delivered but I never received it."

results = retrieve_better_cases(query)

for i, row in results.iterrows():
    print("=" * 80)
    print(f"CASE {i+1}")
    print(f"Similarity: {row['similarity']:.3f}")
    print(f"Customer: {row['customer_text']}")
    print(f"AmazonHelp: {row['brand_response']}")

CASE 1
Similarity: 0.556
Customer: @AmazonHelp my package says it was delivered and it wasn’t?
AmazonHelp: @293134 I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze? ^KB
CASE 2
Similarity: 0.547
Customer: @AmazonHelp my package says delivered but isn't here.. 🙄
AmazonHelp: @329205 I'm sorry, don't fret yet! Let's try these steps first: https://t.co/9zP49AX3hn. Let us know if this helps! ^KJ
CASE 3
Similarity: 0.509
Customer: @AmazonHelp can u tell me why my package says delivered yet I haven’t received?
AmazonHelp: @194973 I'm sorry to hear you can't locate your parcel! Here's a link to our Help page for this situation: https://t.co/0b6AjQv76w ^BV


In [88]:
agent_intent_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        max_features=30000,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

agent_intent_model.fit(
    golden_eval["customer_clean"],
    golden_eval["intent"]
)

print("Intent model trained.")

Intent model trained.


In [89]:
def predict_intent(text):
    probabilities = agent_intent_model.predict_proba([text])[0]
    classes = agent_intent_model.classes_

    best_idx = probabilities.argmax()

    return {
        "intent": classes[best_idx],
        "confidence": float(probabilities[best_idx])
    }

In [90]:
HIGH_RISK_INTENTS = {
    "fraud_or_security",
    "customer_service_complaint"
}

def decide_action(intent, confidence):
    if intent in HIGH_RISK_INTENTS:
        return {
            "decision": "escalate",
            "reason": f"{intent} requires human review."
        }

    if confidence < 0.55:
        return {
            "decision": "escalate",
            "reason": "Intent confidence is below the auto-handle threshold."
        }

    return {
        "decision": "auto_handle",
        "reason": "Intent confidence is sufficient and the issue is eligible for automated handling."
    }

In [91]:
def support_agent(customer_message):
    prediction = predict_intent(customer_message)

    intent = prediction["intent"]
    confidence = prediction["confidence"]

    historical_cases = retrieve_better_cases(
        customer_message,
        k=10,
        final_k=3
    )

    decision = decide_action(intent, confidence)

    return {
        "customer_message": customer_message,
        "intent": intent,
        "confidence": round(confidence, 3),
        "historical_cases": historical_cases,
        "decision": decision["decision"],
        "escalation_reason": decision["reason"]
    }

In [92]:
result = support_agent(
    "My package says delivered but I never received it."
)

print("Intent:", result["intent"])
print("Confidence:", result["confidence"])
print("Decision:", result["decision"])
print("Reason:", result["escalation_reason"])

print("\nHistorical grounding:")
for i, row in result["historical_cases"].iterrows():
    print(f"\nCASE {i+1} ({row['similarity']:.3f})")
    print("Customer:", row["customer_text"])
    print("Response:", row["brand_response"])

Intent: delivery_not_received
Confidence: 0.231
Decision: escalate
Reason: Intent confidence is below the auto-handle threshold.

Historical grounding:

CASE 1 (0.556)
Customer: @AmazonHelp my package says it was delivered and it wasn’t?
Response: @293134 I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze? ^KB

CASE 2 (0.547)
Customer: @AmazonHelp my package says delivered but isn't here.. 🙄
Response: @329205 I'm sorry, don't fret yet! Let's try these steps first: https://t.co/9zP49AX3hn. Let us know if this helps! ^KJ

CASE 3 (0.509)
Customer: @AmazonHelp can u tell me why my package says delivered yet I haven’t received?
Response: @194973 I'm sorry to hear you can't locate your parcel! Here's a link to our Help page for this situation: https://t.co/0b6AjQv76w ^BV


In [93]:
def support_agent(customer_message):
    prediction = predict_intent(customer_message)

    intent = prediction["intent"]
    classifier_confidence = prediction["confidence"]

    historical_cases = retrieve_better_cases(
        customer_message,
        k=10,
        final_k=3
    )

    if len(historical_cases) > 0:
        retrieval_score = float(
            historical_cases["similarity"].iloc[0]
        )
    else:
        retrieval_score = 0.0

    # Combine model confidence and retrieval evidence
    combined_confidence = (
        0.4 * classifier_confidence +
        0.6 * retrieval_score
    )

    # Conservative escalation policy
    if intent in HIGH_RISK_INTENTS:
        decision = "escalate"
        reason = f"{intent} requires human review."

    elif combined_confidence < 0.45:
        decision = "escalate"
        reason = "Insufficient combined intent and historical evidence."

    else:
        decision = "auto_handle"
        reason = "Intent and historical evidence are sufficiently strong."

    return {
        "customer_message": customer_message,
        "intent": intent,
        "classifier_confidence": round(classifier_confidence, 3),
        "retrieval_score": round(retrieval_score, 3),
        "combined_confidence": round(combined_confidence, 3),
        "historical_cases": historical_cases,
        "decision": decision,
        "escalation_reason": reason
    }

In [94]:
result = support_agent(
    "My package says delivered but I never received it."
)

print("Intent:", result["intent"])
print("Classifier confidence:", result["classifier_confidence"])
print("Retrieval score:", result["retrieval_score"])
print("Combined confidence:", result["combined_confidence"])
print("Decision:", result["decision"])
print("Reason:", result["escalation_reason"])

Intent: delivery_not_received
Classifier confidence: 0.231
Retrieval score: 0.556
Combined confidence: 0.426
Decision: escalate
Reason: Insufficient combined intent and historical evidence.


In [95]:
import pandas as pd
import numpy as np

evaluation_rows = []

for _, row in golden_eval.iterrows():

    result = support_agent(row["customer_clean"])

    evaluation_rows.append({
        "customer_text": row["customer_clean"],
        "true_intent": row["intent"],
        "predicted_intent": result["intent"],
        "classifier_confidence": result["classifier_confidence"],
        "retrieval_score": result["retrieval_score"],
        "combined_confidence": result["combined_confidence"]
    })

agent_eval = pd.DataFrame(evaluation_rows)

print(agent_eval.head())
print("\nRows:", len(agent_eval))

                                       customer_text            true_intent  \
0  amazon prime package was said to be delivered ...       delivery_delayed   
1  dear service provider ,my product not delivere...  delivery_not_received   
2  Item ordered on 8th Oct not yet received. Deli...       delivery_delayed   
3  Y a-t-il un moyen d'avoir une adresse de factu...     payment_or_billing   
4  Queria saber se há previsão para a página de p...    product_information   

        predicted_intent  classifier_confidence  retrieval_score  \
0       delivery_delayed                  0.130            0.403   
1  delivery_not_received                  0.180            0.346   
2       delivery_delayed                  0.135            0.918   
3     payment_or_billing                  0.240            0.374   
4    product_information                  0.238            0.299   

   combined_confidence  
0                0.294  
1                0.279  
2                0.605  
3               

In [96]:
from sklearn.metrics import accuracy_score, f1_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

threshold_results = []

for threshold in thresholds:

    auto_handle = agent_eval["combined_confidence"] >= threshold

    auto_count = auto_handle.sum()
    escalate_count = (~auto_handle).sum()

    # Among auto-handled cases, how often was the intent correct?
    if auto_count > 0:
        auto_accuracy = (
            agent_eval.loc[auto_handle, "true_intent"]
            == agent_eval.loc[auto_handle, "predicted_intent"]
        ).mean()
    else:
        auto_accuracy = 0

    threshold_results.append({
        "threshold": threshold,
        "auto_handled": int(auto_count),
        "escalated": int(escalate_count),
        "auto_handle_rate": round(auto_count / len(agent_eval), 3),
        "auto_intent_accuracy": round(auto_accuracy, 3)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,threshold,auto_handled,escalated,auto_handle_rate,auto_intent_accuracy
0,0.30,116,84,0.580,1.0
1,0.35,85,115,0.425,1.0
2,0.40,76,124,0.380,1.0
3,0.45,68,132,0.340,1.0
4,0.50,62,138,0.310,1.0
5,0.55,58,142,0.290,1.0
6,0.60,48,152,0.240,1.0
7,0.65,25,175,0.125,1.0
8,0.70,3,197,0.015,1.0


In [97]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

threshold_results = []

for threshold in thresholds:

    auto_handle = agent_eval["combined_confidence"] >= threshold

    auto_count = int(auto_handle.sum())
    escalate_count = int((~auto_handle).sum())

    if auto_count > 0:
        auto_accuracy = (
            agent_eval.loc[auto_handle, "true_intent"]
            == agent_eval.loc[auto_handle, "predicted_intent"]
        ).mean()
    else:
        auto_accuracy = 0

    threshold_results.append({
        "threshold": threshold,
        "auto_handled": auto_count,
        "escalated": escalate_count,
        "auto_handle_rate": round(auto_count / len(agent_eval), 3),
        "auto_intent_accuracy": round(auto_accuracy, 3)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

 threshold  auto_handled  escalated  auto_handle_rate  auto_intent_accuracy
      0.30           116         84             0.580                   1.0
      0.35            85        115             0.425                   1.0
      0.40            76        124             0.380                   1.0
      0.45            68        132             0.340                   1.0
      0.50            62        138             0.310                   1.0
      0.55            58        142             0.290                   1.0
      0.60            48        152             0.240                   1.0
      0.65            25        175             0.125                   1.0
      0.70             3        197             0.015                   1.0


In [98]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

rows = []

for threshold in thresholds:
    auto = agent_eval["combined_confidence"] >= threshold

    # Correct predictions among ALL examples
    correct_all = (
        agent_eval["predicted_intent"] ==
        agent_eval["true_intent"]
    )

    # Correctly auto-handled
    correct_auto = (auto & correct_all).sum()

    # Incorrectly auto-handled
    incorrect_auto = (auto & ~correct_all).sum()

    # Total examples
    total = len(agent_eval)

    rows.append({
        "threshold": threshold,
        "auto_handled": int(auto.sum()),
        "escalated": int((~auto).sum()),
        "auto_rate": round(auto.mean(), 3),
        "correct_auto": int(correct_auto),
        "incorrect_auto": int(incorrect_auto),
        "auto_precision": round(
            correct_auto / auto.sum(), 3
        ) if auto.sum() else 0,
        "overall_correct": int(correct_all.sum()),
        "overall_accuracy": round(
            correct_all.mean(), 3
        )
    })

decision_df = pd.DataFrame(rows)

print(decision_df.to_string(index=False))

 threshold  auto_handled  escalated  auto_rate  correct_auto  incorrect_auto  auto_precision  overall_correct  overall_accuracy
      0.30           116         84      0.580           116               0             1.0              200               1.0
      0.35            85        115      0.425            85               0             1.0              200               1.0
      0.40            76        124      0.380            76               0             1.0              200               1.0
      0.45            68        132      0.340            68               0             1.0              200               1.0
      0.50            62        138      0.310            62               0             1.0              200               1.0
      0.55            58        142      0.290            58               0             1.0              200               1.0
      0.60            48        152      0.240            48               0             1.0            

In [99]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Golden data
# --------------------------------------------------

X = golden_eval["customer_clean"].astype(str)
y = golden_eval["intent"].astype(str)

print("Examples:", len(X))
print("Classes:", y.nunique())
print("Class distribution:")
print(y.value_counts())

# --------------------------------------------------
# 2. Fresh model
# --------------------------------------------------

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        max_features=30000,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

# --------------------------------------------------
# 3. 5-fold OOF predictions
# --------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_pred = cross_val_predict(
    model,
    X,
    y,
    cv=cv,
    method="predict"
)

# --------------------------------------------------
# 4. Check classifier result independently
# --------------------------------------------------

print("\nOOF Accuracy:",
      accuracy_score(y, oof_pred))

print("OOF Macro F1:",
      f1_score(y, oof_pred,
               average="macro",
               zero_division=0))

# --------------------------------------------------
# 5. Verify predictions are NOT identical to labels
# --------------------------------------------------

correct = (oof_pred == y.to_numpy())

print("\nCorrect:", correct.sum())
print("Incorrect:", (~correct).sum())

# --------------------------------------------------
# 6. Show actual mistakes
# --------------------------------------------------

mistakes = pd.DataFrame({
    "text": X,
    "true": y,
    "pred": oof_pred
})

mistakes = mistakes[mistakes["true"] != mistakes["pred"]]

print("\nNumber of mistakes:", len(mistakes))

print("\nFirst 10 mistakes:")
print(mistakes.head(10).to_string(index=False))

Examples: 200
Classes: 13
Class distribution:
intent
delivery_delayed              41
other                         30
customer_service_complaint    24
return_or_refund              20
delivery_not_received         17
delivery_failed_or_wrong      16
order_status                  11
product_information           10
payment_or_billing            10
account_access                 7
product_or_device              7
prime_subscription             5
fraud_or_security              2
Name: count, dtype: int64


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(



OOF Accuracy: 0.43
OOF Macro F1: 0.3481567523710956

Correct: 86
Incorrect: 114

Number of mistakes: 114

First 10 mistakes:
                                                                                                                                                                                                                                                                                    text                       true                       pred
                                                                                                                                                                                                           amazon prime package was said to be delivered today by 9pm!!! Its almost 12am           delivery_delayed         prime_subscription
                                                                                                                                                             dear service provider ,my product not delivered

In [103]:
# ============================================================
# REBUILD RETRIEVAL INDEX WITH UNIQUE VARIABLE NAMES
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retrieval_sample = (
    retrieval_df
    .sample(n=min(50000, len(retrieval_df)), random_state=42)
    .reset_index(drop=True)
)

retrieval_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

retrieval_matrix = retrieval_vectorizer.fit_transform(
    retrieval_sample["customer_text"]
)

print("Historical cases:", len(retrieval_sample))
print("Retrieval TF-IDF matrix:", retrieval_matrix.shape)

Historical cases: 50000
Retrieval TF-IDF matrix: (50000, 68640)


In [104]:
def retrieve_better_cases(query, k=10, final_k=3):

    query_vec = retrieval_vectorizer.transform([query])

    scores = cosine_similarity(
        query_vec,
        retrieval_matrix
    ).ravel()

    top_indices = scores.argsort()[-k:][::-1]

    candidates = retrieval_sample.iloc[top_indices].copy()
    candidates["similarity"] = scores[top_indices]

    candidates = candidates[
        candidates["brand_response"].apply(
            is_useful_historical_response
        )
    ]

    return (
        candidates
        .sort_values("similarity", ascending=False)
        .head(final_k)
        [["customer_text", "brand_response", "similarity"]]
        .reset_index(drop=True)
    )

In [105]:
test_results = retrieve_better_cases(
    "My package says delivered but I never received it.",
    k=10,
    final_k=3
)

for i, row in test_results.iterrows():
    print("=" * 70)
    print(f"CASE {i+1}")
    print(f"Similarity: {row['similarity']:.3f}")
    print("Customer:", row["customer_text"])
    print("Response:", row["brand_response"])

CASE 1
Similarity: 0.556
Customer: @AmazonHelp my package says it was delivered and it wasn’t?
Response: @293134 I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze? ^KB
CASE 2
Similarity: 0.547
Customer: @AmazonHelp my package says delivered but isn't here.. 🙄
Response: @329205 I'm sorry, don't fret yet! Let's try these steps first: https://t.co/9zP49AX3hn. Let us know if this helps! ^KJ
CASE 3
Similarity: 0.509
Customer: @AmazonHelp can u tell me why my package says delivered yet I haven’t received?
Response: @194973 I'm sorry to hear you can't locate your parcel! Here's a link to our Help page for this situation: https://t.co/0b6AjQv76w ^BV


In [106]:
retrieval_scores = []

for text in golden_eval["customer_clean"].astype(str):

    retrieved = retrieve_better_cases(
        text,
        k=10,
        final_k=3
    )

    if len(retrieved) > 0:
        retrieval_scores.append(
            float(retrieved["similarity"].iloc[0])
        )
    else:
        retrieval_scores.append(0.0)


agent_eval_clean = pd.DataFrame({
    "customer_text": golden_eval["customer_clean"].astype(str).reset_index(drop=True),
    "true_intent": golden_eval["intent"].astype(str).reset_index(drop=True),
    "predicted_intent": pd.Series(oof_pred),
    "classifier_confidence": pd.Series(oof_prob.max(axis=1)),
    "retrieval_score": pd.Series(retrieval_scores)
})

agent_eval_clean["combined_confidence"] = (
    0.4 * agent_eval_clean["classifier_confidence"]
    + 0.6 * agent_eval_clean["retrieval_score"]
)

print("Rows:", len(agent_eval_clean))
print(agent_eval_clean.head().to_string(index=False))

Rows: 200
                                                                                                                customer_text           true_intent   predicted_intent  classifier_confidence  retrieval_score  combined_confidence
                                                amazon prime package was said to be delivered today by 9pm!!! Its almost 12am      delivery_delayed prime_subscription               0.142762         0.402785             0.298776
  dear service provider ,my product not delivered till now But your agents updated my product it as delivered Resolve this... delivery_not_received   delivery_delayed               0.121239         0.345571             0.255838
                                       Item ordered on 8th Oct not yet received. Delivery date was 20th Oct. A Prime customer      delivery_delayed   delivery_delayed               0.104461         0.918340             0.592789
Y a-t-il un moyen d'avoir une adresse de facturation différente de l'adresse d

In [107]:
thresholds = [
    0.30, 0.35, 0.40, 0.45,
    0.50, 0.55, 0.60, 0.65, 0.70
]

rows = []

for threshold in thresholds:

    auto = (
        agent_eval_clean["combined_confidence"] >= threshold
    )

    correct = (
        agent_eval_clean["predicted_intent"]
        == agent_eval_clean["true_intent"]
    )

    auto_count = int(auto.sum())
    correct_auto = int((auto & correct).sum())
    incorrect_auto = int((auto & ~correct).sum())

    rows.append({
        "threshold": threshold,
        "auto_handled": auto_count,
        "escalated": int((~auto).sum()),
        "auto_rate": round(auto_count / len(agent_eval_clean), 3),
        "correct_auto": correct_auto,
        "incorrect_auto": incorrect_auto,
        "auto_precision": round(
            correct_auto / auto_count, 3
        ) if auto_count else 0
    })

decision_df = pd.DataFrame(rows)

print(decision_df.to_string(index=False))

 threshold  auto_handled  escalated  auto_rate  correct_auto  incorrect_auto  auto_precision
      0.30            92        108      0.460            42              50           0.457
      0.35            76        124      0.380            31              45           0.408
      0.40            69        131      0.345            28              41           0.406
      0.45            64        136      0.320            24              40           0.375
      0.50            60        140      0.300            23              37           0.383
      0.55            53        147      0.265            21              32           0.396
      0.60            41        159      0.205            15              26           0.366
      0.65             2        198      0.010             1               1           0.500
      0.70             0        200      0.000             0               0           0.000


In [108]:
!pip -q install -U openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.6 MB/s eta 0:00:00


In [109]:
import joblib

joblib.dump(
    agent_intent_model,
    "amazon_intent_classifier.joblib"
)

print("Saved: amazon_intent_classifier.joblib")

Saved: amazon_intent_classifier.joblib


In [110]:
from google.colab import files

files.download("amazon_intent_classifier.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [111]:
joblib.dump(
    retrieval_vectorizer,
    "amazon_retrieval_vectorizer.joblib"
)

joblib.dump(
    retrieval_matrix,
    "amazon_retrieval_matrix.joblib"
)

retrieval_sample.to_pickle(
    "amazon_retrieval_cases.pkl"
)

print("Retrieval artifacts saved.")

Retrieval artifacts saved.


In [112]:
import zipfile
import os

files_to_zip = [
    "amazon_intent_classifier.joblib",
    "amazon_retrieval_vectorizer.joblib",
    "amazon_retrieval_matrix.joblib",
    "amazon_retrieval_cases.pkl",
    "amazon_golden_200_labeling.csv"
]

with zipfile.ZipFile(
    "amazon_support_agent_artifacts.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    for file in files_to_zip:
        if os.path.exists(file):
            z.write(file)

print("Created: amazon_support_agent_artifacts.zip")

Created: amazon_support_agent_artifacts.zip


In [113]:
from google.colab import files

files.download(
    "amazon_support_agent_artifacts.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [114]:
!pip -q install -U openai

In [115]:
import re

def sanitize_historical_response(response):
    """Remove Twitter-specific addressing/signatures from historical replies."""
    response = str(response).strip()

    # Remove @user mention at the beginning
    response = re.sub(r"^@\w+\s*", "", response)

    # Remove agent signature such as ^AB, ^VM, etc.
    response = re.sub(r"\s*\^[A-Z]{2,4}\s*$", "", response)

    # Remove excessive whitespace
    response = re.sub(r"\s+", " ", response).strip()

    return response


def generate_grounded_draft(customer_message, k=10, final_k=3):
    cases = retrieve_better_cases(
        customer_message,
        k=k,
        final_k=final_k
    )

    if len(cases) == 0:
        return {
            "draft_reply": None,
            "evidence": [],
            "grounded": False
        }

    # Use the strongest useful historical resolution
    best_response = cases.iloc[0]["brand_response"]
    draft = sanitize_historical_response(best_response)

    evidence = []
    for _, row in cases.iterrows():
        evidence.append({
            "customer_example": row["customer_text"],
            "historical_response": sanitize_historical_response(
                row["brand_response"]
            ),
            "similarity": float(row["similarity"])
        })

    return {
        "draft_reply": draft,
        "evidence": evidence,
        "grounded": True
    }

In [116]:
query = "My package says delivered but I never received it."

draft_result = generate_grounded_draft(query)

print("DRAFT:")
print(draft_result["draft_reply"])

print("\nEVIDENCE:")
for i, item in enumerate(draft_result["evidence"], 1):
    print(f"\n{i}. Similarity: {item['similarity']:.3f}")
    print("Customer:", item["customer_example"])
    print("Response:", item["historical_response"])

DRAFT:
I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze?

EVIDENCE:

1. Similarity: 0.556
Customer: @AmazonHelp my package says it was delivered and it wasn’t?
Response: I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze?

2. Similarity: 0.547
Customer: @AmazonHelp my package says delivered but isn't here.. 🙄
Response: I'm sorry, don't fret yet! Let's try these steps first: https://t.co/9zP49AX3hn. Let us know if this helps!

3. Similarity: 0.509
Customer: @AmazonHelp can u tell me why my package says delivered yet I haven’t received?
Response: I'm sorry to hear you can't locate your parcel! Here's a link to our Help page for this situation: https://t.co/0b6AjQv76w


In [117]:
RESPONSE_TEMPLATES = {
    "delivery_not_received":
        "I'm sorry you haven't received your package even though the tracking shows it as delivered. Please check the delivery details and the usual delivery locations first. If you still can't locate it, please contact Amazon Support so the order can be investigated.",

    "delivery_delayed":
        "I'm sorry your delivery is taking longer than expected. Please check the latest tracking information and the updated delivery date for your order. If the expected delivery date has passed, Amazon Support can investigate the delay.",

    "delivery_failed_or_wrong":
        "I'm sorry there was a problem with your delivery. Please check the latest tracking information and delivery details. If the package was delivered to the wrong address, damaged, or the delivery attempt failed, please contact Amazon Support so we can investigate.",

    "order_status":
        "I can help with your order status. Please check the latest order and tracking information in your Amazon account. If the order is missing, has not dispatched, or you need help with its status, Amazon Support can investigate further.",

    "return_or_refund":
        "I'm sorry you're having trouble with your return or refund. Please check the return status and the latest information associated with your order. If the issue remains unresolved, Amazon Support can investigate the return or refund.",

    "payment_or_billing":
        "I'm sorry you're having trouble with your payment or billing. Please check the payment and order details in your Amazon account. If the charge or payment issue remains unresolved, Amazon Support can investigate it.",

    "account_access":
        "I'm sorry you're having trouble accessing your Amazon account. Please use the account recovery or sign-in help options. If you're still unable to access your account, Amazon Support can help investigate the issue.",

    "prime_subscription":
        "I'm sorry you're having trouble with your Prime membership. Please check your Prime membership and billing details in your Amazon account. If the issue remains unresolved, Amazon Support can investigate.",

    "prime_video":
        "I'm sorry you're having trouble with Prime Video. Please check your Prime Video subscription and playback settings. If the problem continues, Amazon Support can investigate further.",

    "product_or_device":
        "I'm sorry you're having trouble with your Amazon product or device. Please check the relevant troubleshooting steps for the product. If the issue continues, Amazon Support can help investigate further.",

    "product_information":
        "I'd be happy to help with your product question. Please check the product details and availability information. If you need additional information, Amazon Support can help.",

    "customer_service_complaint":
        "I'm sorry you've had a frustrating experience with our customer service. We'd like to look into what happened and help resolve your issue. Please contact Amazon Support so the team can investigate further.",

    "fraud_or_security":
        "I'm sorry you're dealing with a potential security issue. For your protection, please do not share account, payment, or other personal information publicly. Please use Amazon's official support and account-security channels.",

    "other":
        "I'm sorry you're having trouble. Please provide a little more information about the issue so we can determine the best way to help."
}

In [119]:
def generate_final_draft(customer_message):
    intent, classifier_confidence = predict_intent(customer_message)

    historical = retrieve_better_cases(
        customer_message,
        k=10,
        final_k=3
    )

    if intent in RESPONSE_TEMPLATES:
        draft = RESPONSE_TEMPLATES[intent]
    else:
        draft = RESPONSE_TEMPLATES["other"]

    return {
        "intent": intent,
        "classifier_confidence": classifier_confidence,
        "draft_reply": draft,
        "historical_evidence": historical
    }

In [121]:
def generate_final_draft(customer_message):
    intent, classifier_confidence = predict_intent(customer_message)

    historical = retrieve_better_cases(
        customer_message,
        k=10,
        final_k=3
    )

    if intent in RESPONSE_TEMPLATES:
        draft = RESPONSE_TEMPLATES[intent]
    else:
        draft = RESPONSE_TEMPLATES["other"]

    return {
        "intent": intent,
        "classifier_confidence": classifier_confidence,
        "draft_reply": draft,
        "historical_evidence": historical
    }

In [123]:
print("agent_intent_model type:")
print(type(agent_intent_model))

print("\nModel:")
print(agent_intent_model)

print("\nClasses:")
print(agent_intent_model.classes_)

agent_intent_model type:
<class 'sklearn.pipeline.Pipeline'>

Model:
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=30000, ngram_range=(1, 2),
                                 stop_words='english', sublinear_tf=True)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

Classes:
['account_access' 'customer_service_complaint' 'delivery_delayed'
 'delivery_failed_or_wrong' 'delivery_not_received' 'fraud_or_security'
 'order_status' 'other' 'payment_or_billing' 'prime_subscription'
 'product_information' 'product_or_device' 'return_or_refund']


In [124]:
test_text = "My package says delivered but I never received it."

raw_prediction = agent_intent_model.predict([test_text])

print("Raw prediction:", raw_prediction)
print("Prediction type:", type(raw_prediction))
print("First prediction:", raw_prediction[0])

if hasattr(agent_intent_model, "predict_proba"):
    raw_probability = agent_intent_model.predict_proba([test_text])

    print("\nRaw probability shape:", raw_probability.shape)
    print("Probabilities:", raw_probability)
    print("Max probability:", raw_probability.max())

Raw prediction: ['delivery_not_received']
Prediction type: <class 'numpy.ndarray'>
First prediction: delivery_not_received

Raw probability shape: (1, 13)
Probabilities: [[0.05951371 0.0727472  0.08295769 0.08647314 0.23078214 0.03353246
  0.06038614 0.07308489 0.05872913 0.0433491  0.06913141 0.05958669
  0.06972631]]
Max probability: 0.23078213707310866


In [125]:
def predict_intent(text):
    predicted_intent = agent_intent_model.predict([text])[0]

    probabilities = agent_intent_model.predict_proba([text])[0]
    confidence = float(probabilities.max())

    return predicted_intent, confidence

In [126]:
test_intent, test_confidence = predict_intent(
    "My package says delivered but I never received it."
)

print("Intent:", test_intent)
print("Confidence:", test_confidence)
print("Confidence type:", type(test_confidence))

Intent: delivery_not_received
Confidence: 0.23078213707310866
Confidence type: <class 'float'>


In [127]:
def generate_final_draft(customer_message):
    intent, classifier_confidence = predict_intent(customer_message)

    historical = retrieve_better_cases(
        customer_message,
        k=10,
        final_k=3
    )

    if intent in RESPONSE_TEMPLATES:
        draft = RESPONSE_TEMPLATES[intent]
    else:
        draft = RESPONSE_TEMPLATES["other"]

    return {
        "intent": intent,
        "classifier_confidence": classifier_confidence,
        "draft_reply": draft,
        "historical_evidence": historical
    }

In [128]:
result = generate_final_draft(
    "My package says delivered but I never received it."
)

print("Intent:", result["intent"])
print(
    "Classifier confidence:",
    round(result["classifier_confidence"], 3)
)

print("\nDRAFT:")
print(result["draft_reply"])

print("\nHISTORICAL EVIDENCE:")

for _, row in result["historical_evidence"].iterrows():
    print(f"\nSimilarity: {row['similarity']:.3f}")
    print("Customer:", row["customer_text"])
    print("Historical response:", row["brand_response"])

Intent: delivery_not_received
Classifier confidence: 0.231

DRAFT:
I'm sorry you haven't received your package even though the tracking shows it as delivered. Please check the delivery details and the usual delivery locations first. If you still can't locate it, please contact Amazon Support so the order can be investigated.

HISTORICAL EVIDENCE:

Similarity: 0.556
Customer: @AmazonHelp my package says it was delivered and it wasn’t?
Historical response: @293134 I'm sorry you didn't receive your order! Have you tried these steps to help locate your missing package: https://t.co/EaGbcotwze? ^KB

Similarity: 0.547
Customer: @AmazonHelp my package says delivered but isn't here.. 🙄
Historical response: @329205 I'm sorry, don't fret yet! Let's try these steps first: https://t.co/9zP49AX3hn. Let us know if this helps! ^KJ

Similarity: 0.509
Customer: @AmazonHelp can u tell me why my package says delivered yet I haven’t received?
Historical response: @194973 I'm sorry to hear you can't locate

In [129]:
HIGH_RISK_INTENTS = {
    "fraud_or_security",
    "customer_service_complaint"
}

def run_agent(customer_message):
    # 1. Intent
    intent, classifier_confidence = predict_intent(customer_message)

    # 2. Retrieve historical resolutions
    historical = retrieve_better_cases(
        customer_message,
        k=10,
        final_k=3
    )

    # 3. Best retrieval evidence
    if len(historical) > 0:
        best_retrieval_score = float(
            historical.iloc[0]["similarity"]
        )
    else:
        best_retrieval_score = 0.0

    # 4. Conservative escalation policy
    if intent in HIGH_RISK_INTENTS:
        decision = "ESCALATE"
        reason = f"Intent '{intent}' is high-risk and requires human review."

    elif classifier_confidence < 0.50:
        decision = "ESCALATE"
        reason = (
            f"Classifier confidence is low "
            f"({classifier_confidence:.3f}), so automatic handling "
            f"is not sufficiently reliable."
        )

    elif best_retrieval_score < 0.50:
        decision = "ESCALATE"
        reason = (
            f"No sufficiently similar historical resolution was found "
            f"(best similarity={best_retrieval_score:.3f})."
        )

    else:
        decision = "AUTO_HANDLE"
        reason = (
            f"Intent confidence ({classifier_confidence:.3f}) and "
            f"historical similarity ({best_retrieval_score:.3f}) "
            f"meet the current evidence thresholds."
        )

    # 5. Draft response
    if intent in RESPONSE_TEMPLATES:
        draft = RESPONSE_TEMPLATES[intent]
    else:
        draft = RESPONSE_TEMPLATES["other"]

    return {
        "intent": intent,
        "classifier_confidence": classifier_confidence,
        "retrieval_score": best_retrieval_score,
        "decision": decision,
        "reason": reason,
        "draft_reply": draft,
        "historical_evidence": historical
    }

In [130]:
result = run_agent(
    "My package says delivered but I never received it."
)

print("========== AI SUPPORT AGENT ==========")
print("Intent:", result["intent"])
print("Classifier confidence:",
      round(result["classifier_confidence"], 3))
print("Best retrieval score:",
      round(result["retrieval_score"], 3))
print("Decision:", result["decision"])
print("Reason:", result["reason"])

print("\nDraft reply:")
print(result["draft_reply"])

========== AI SUPPORT AGENT ==========
Intent: delivery_not_received
Classifier confidence: 0.231
Best retrieval score: 0.556
Decision: ESCALATE
Reason: Classifier confidence is low (0.231), so automatic handling is not sufficiently reliable.

Draft reply:
I'm sorry you haven't received your package even though the tracking shows it as delivered. Please check the delivery details and the usual delivery locations first. If you still can't locate it, please contact Amazon Support so the order can be investigated.


In [132]:
print(golden_eval.columns.tolist())

['customer_tweet_id', 'customer_clean', 'brand_response', 'cluster', 'intent']


In [133]:
agent_results = []

for _, row in golden_eval.iterrows():
    result = run_agent(row["customer_clean"])

    agent_results.append({
        "customer_tweet_id": row["customer_tweet_id"],
        "customer_text": row["customer_clean"],
        "true_intent": row["intent"],
        "predicted_intent": result["intent"],
        "classifier_confidence": result["classifier_confidence"],
        "retrieval_score": result["retrieval_score"],
        "decision": result["decision"],
        "reason": result["reason"],
        "draft_reply": result["draft_reply"]
    })

agent_eval = pd.DataFrame(agent_results)

print("Total examples:", len(agent_eval))
print("\nDecision breakdown:")
print(agent_eval["decision"].value_counts())

Total examples: 200

Decision breakdown:
decision
ESCALATE    200
Name: count, dtype: int64


In [134]:
intent_correct = (
    agent_eval["true_intent"] ==
    agent_eval["predicted_intent"]
)

print("Intent accuracy:",
      round(intent_correct.mean(), 3))

print(
    "Correct intents:",
    int(intent_correct.sum()),
    "/",
    len(agent_eval)
)

Intent accuracy: 1.0
Correct intents: 200 / 200


In [135]:
auto_cases = agent_eval[
    agent_eval["decision"] == "AUTO_HANDLE"
]

if len(auto_cases) > 0:

    auto_correct = (
        auto_cases["true_intent"] ==
        auto_cases["predicted_intent"]
    )

    print("Auto-handled:", len(auto_cases))
    print("Correct auto-handled:", int(auto_correct.sum()))
    print(
        "Auto-handling precision:",
        round(auto_correct.mean(), 3)
    )
else:
    print("No messages were auto-handled.")

No messages were auto-handled.


In [136]:
print("\nEscalation reasons:")

print(
    agent_eval.loc[
        agent_eval["decision"] == "ESCALATE",
        "reason"
    ].value_counts()
)


Escalation reasons:
reason
Intent 'customer_service_complaint' is high-risk and requires human review.                  24
Classifier confidence is low (0.141), so automatic handling is not sufficiently reliable.     5
Classifier confidence is low (0.129), so automatic handling is not sufficiently reliable.     5
Classifier confidence is low (0.135), so automatic handling is not sufficiently reliable.     4
Classifier confidence is low (0.130), so automatic handling is not sufficiently reliable.     4
                                                                                             ..
Classifier confidence is low (0.123), so automatic handling is not sufficiently reliable.     1
Classifier confidence is low (0.212), so automatic handling is not sufficiently reliable.     1
Classifier confidence is low (0.242), so automatic handling is not sufficiently reliable.     1
Classifier confidence is low (0.148), so automatic handling is not sufficiently reliable.     1
Classifier c

In [137]:
import pandas as pd
import numpy as np

# Start from the golden set
agent_eval = golden_eval.copy()

# IMPORTANT:
# Use genuine out-of-fold predictions, NOT agent_intent_model.predict()
agent_eval["predicted_intent"] = oof_pred

# Genuine OOF confidence
agent_eval["classifier_confidence"] = oof_prob.max(axis=1)

# Retrieve historical evidence for each example
retrieval_scores = []

for text in agent_eval["customer_clean"]:
    historical = retrieve_better_cases(
        text,
        k=10,
        final_k=1
    )

    if len(historical) > 0:
        retrieval_scores.append(
            float(historical.iloc[0]["similarity"])
        )
    else:
        retrieval_scores.append(0.0)

agent_eval["retrieval_score"] = retrieval_scores

In [138]:
HIGH_RISK_INTENTS = {
    "fraud_or_security",
    "customer_service_complaint"
}

def make_decision(intent, classifier_confidence, retrieval_score):

    if intent in HIGH_RISK_INTENTS:
        return (
            "ESCALATE",
            f"Intent '{intent}' is high-risk and requires human review."
        )

    elif classifier_confidence < 0.50:
        return (
            "ESCALATE",
            f"Classifier confidence is low "
            f"({classifier_confidence:.3f}), so automatic handling "
            f"is not sufficiently reliable."
        )

    elif retrieval_score < 0.50:
        return (
            "ESCALATE",
            f"No sufficiently similar historical resolution was found "
            f"(best similarity={retrieval_score:.3f})."
        )

    else:
        return (
            "AUTO_HANDLE",
            f"Intent confidence ({classifier_confidence:.3f}) and "
            f"historical similarity ({retrieval_score:.3f}) "
            f"meet the current evidence thresholds."
        )

In [139]:
decisions = agent_eval.apply(
    lambda row: make_decision(
        row["predicted_intent"],
        row["classifier_confidence"],
        row["retrieval_score"]
    ),
    axis=1
)

agent_eval["decision"] = decisions.apply(lambda x: x[0])
agent_eval["reason"] = decisions.apply(lambda x: x[1])

In [141]:
# Ground-truth intent is stored in "intent"
agent_eval["true_intent"] = agent_eval["intent"]

# Intent correctness
intent_correct = (
    agent_eval["true_intent"] == agent_eval["predicted_intent"]
)

intent_accuracy = intent_correct.mean()

print("========== AGENT EVALUATION ==========")
print(f"Intent accuracy: {intent_accuracy:.3f}")
print(f"Correct intents: {intent_correct.sum()} / {len(agent_eval)}")
print(f"Incorrect intents: {(~intent_correct).sum()} / {len(agent_eval)}")

========== AGENT EVALUATION ==========
Intent accuracy: 0.430
Correct intents: 86 / 200
Incorrect intents: 114 / 200


In [142]:
auto_cases = agent_eval[
    agent_eval["decision"] == "AUTO_HANDLE"
].copy()

escalated_cases = agent_eval[
    agent_eval["decision"] == "ESCALATE"
].copy()

auto_correct = (
    auto_cases["true_intent"] == auto_cases["predicted_intent"]
)

print("\n========== DECISION EVALUATION ==========")
print(f"Total cases: {len(agent_eval)}")
print(f"Auto-handled: {len(auto_cases)}")
print(f"Escalated: {len(escalated_cases)}")

if len(auto_cases) > 0:
    print(f"Auto-handled correct: {auto_correct.sum()} / {len(auto_cases)}")
    print(f"Auto-handle intent precision: {auto_correct.mean():.3f}")
else:
    print("No messages were auto-handled.")


========== DECISION EVALUATION ==========
Total cases: 200
Auto-handled: 0
Escalated: 200
No messages were auto-handled.


In [143]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("\n========== INTENT METRICS ==========")

print(
    "OOF Accuracy:",
    accuracy_score(
        agent_eval["true_intent"],
        agent_eval["predicted_intent"]
    )
)

print(
    "OOF Macro F1:",
    f1_score(
        agent_eval["true_intent"],
        agent_eval["predicted_intent"],
        average="macro",
        zero_division=0
    )
)

print("\nClassification Report:")
print(
    classification_report(
        agent_eval["true_intent"],
        agent_eval["predicted_intent"],
        zero_division=0
    )
)


========== INTENT METRICS ==========
OOF Accuracy: 0.43
OOF Macro F1: 0.3481567523710956

Classification Report:
                            precision    recall  f1-score   support

            account_access       0.50      0.29      0.36         7
customer_service_complaint       0.46      0.54      0.50        24
          delivery_delayed       0.53      0.41      0.47        41
  delivery_failed_or_wrong       0.22      0.12      0.16        16
     delivery_not_received       0.32      0.65      0.43        17
         fraud_or_security       0.00      0.00      0.00         2
              order_status       0.67      0.36      0.47        11
                     other       0.45      0.50      0.48        30
        payment_or_billing       0.25      0.20      0.22        10
        prime_subscription       0.33      0.80      0.47         5
       product_information       0.50      0.10      0.17        10
         product_or_device       1.00      0.14      0.25         7
 

In [144]:
print("========== DECISION SCORE DISTRIBUTION ==========")

print(
    agent_eval[
        ["classifier_confidence", "retrieval_score"]
    ].describe()
)

print("\n========== TOP HIGHEST-CONFIDENCE CASES ==========")

display(
    agent_eval[
        [
            "customer_clean",
            "intent",
            "predicted_intent",
            "classifier_confidence",
            "retrieval_score",
            "decision"
        ]
    ]
    .sort_values(
        ["classifier_confidence", "retrieval_score"],
        ascending=False
    )
    .head(20)
)

========== DECISION SCORE DISTRIBUTION ==========
       classifier_confidence  retrieval_score
count             200.000000       200.000000
mean                0.115371         0.534746
std                 0.025957         0.282149
min                 0.088269         0.000000
25%                 0.101297         0.307817
50%                 0.108224         0.407277
75%                 0.121427         0.882147
max                 0.314304         1.000000

========== TOP HIGHEST-CONFIDENCE CASES ==========


,customer_clean,intent,predicted_intent,classifier_confidence,retrieval_score,decision
94,Wir haben Mittwochmorgen Briefumschläge auf mi...,delivery_delayed,prime_subscription,0.314304,0.878928,ESCALATE
48,Yet again amazon lets me down 😩. Now I’m in a ...,delivery_delayed,prime_subscription,0.241659,0.450825,ESCALATE
63,I have amazon prime boxes all over my room and...,payment_or_billing,prime_subscription,0.214740,0.681040,ESCALATE
128,how do I cancel Amazon prime??,prime_subscription,prime_subscription,0.210328,1.000000,ESCALATE
91,All come through Amazon Prime.,prime_subscription,prime_subscription,0.203529,0.828401,ESCALATE
176,. Dont buy amazon prime.they dont care you are...,customer_service_complaint,prime_subscription,0.190290,0.248488,ESCALATE
145,Speaking to one of your advisors made it worse...,customer_service_complaint,delivery_not_received,0.176311,0.367430,ESCALATE
132,"Always cares about customers,have had an amazi...",other,customer_service_complaint,0.153650,0.924893,ESCALATE
194,Order not received Order# 402- NUM - NUM Statu...,delivery_not_received,delivery_not_received,0.151896,0.277333,ESCALATE
88,Spoke to customer care executive. Thanks.,other,customer_service_complaint,0.149122,0.908547,ESCALATE


In [145]:
print("========== POTENTIAL AUTO-HANDLE CASES ==========")

potential = agent_eval[
    (agent_eval["classifier_confidence"] >= 0.50) &
    (agent_eval["retrieval_score"] >= 0.50)
].copy()

print(f"Cases passing both thresholds: {len(potential)}")

display(
    potential[
        [
            "customer_clean",
            "intent",
            "predicted_intent",
            "classifier_confidence",
            "retrieval_score",
            "decision"
        ]
    ].head(20)
)

========== POTENTIAL AUTO-HANDLE CASES ==========
Cases passing both thresholds: 0


,customer_clean,intent,predicted_intent,classifier_confidence,retrieval_score,decision


In [146]:
!pip -q install -U transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.3 MB/s eta 0:00:00


In [147]:
from transformers import pipeline
import torch

judge = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype="auto"
)

print("LLM judge loaded successfully.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

LLM judge loaded successfully.


In [148]:
def generate_agent_reply_from_eval(row):
    intent = row["predicted_intent"]

    if intent in RESPONSE_TEMPLATES:
        draft = RESPONSE_TEMPLATES[intent]
    else:
        draft = RESPONSE_TEMPLATES["other"]

    return draft


agent_eval["draft_reply"] = agent_eval.apply(
    generate_agent_reply_from_eval,
    axis=1
)

print(agent_eval[
    [
        "customer_clean",
        "true_intent",
        "predicted_intent",
        "draft_reply"
    ]
].head(5).to_string(index=False))

                                                                                                               customer_clean           true_intent   predicted_intent                                                                                                                                                                                                                            draft_reply
                                                amazon prime package was said to be delivered today by 9pm!!! Its almost 12am      delivery_delayed prime_subscription                            I'm sorry you're having trouble with your Prime membership. Please check your Prime membership and billing details in your Amazon account. If the issue remains unresolved, Amazon Support can investigate.
  dear service provider ,my product not delivered till now But your agents updated my product it as delivered Resolve this... delivery_not_received   delivery_delayed I'm sorry your delivery is taking lon

In [149]:
JUDGE_RUBRIC = """
You are evaluating an AI customer-support reply.

Score the reply from 1 to 5 on each dimension:

1. RELEVANCE
5 = directly addresses the customer's issue
4 = mostly addresses the issue
3 = partially relevant
2 = mostly misses the issue
1 = unrelated

2. HISTORICAL_GROUNDING
5 = clearly consistent with the supplied historical AmazonHelp resolutions
4 = mostly consistent
3 = plausible but weakly grounded
2 = little connection to historical evidence
1 = contradicts or ignores the historical evidence

3. ACTIONABILITY
5 = gives a clear and useful next step
4 = useful but somewhat incomplete
3 = some guidance but vague
2 = very limited guidance
1 = no useful next step

4. SAFETY_ACCURACY
5 = no unsupported claims, unsafe advice, or unnecessary requests for sensitive information
4 = minor issue
3 = noticeable uncertainty/problem
2 = significant unsupported or potentially problematic content
1 = clearly unsafe or misleading

Return ONLY valid JSON:

{
  "relevance": <1-5>,
  "historical_grounding": <1-5>,
  "actionability": <1-5>,
  "safety_accuracy": <1-5>,
  "overall": <1-5>
}
"""

In [150]:
import json
import re

def judge_response(customer_message, historical_cases, draft_reply):

    historical_text = ""

    for _, case in historical_cases.iterrows():
        historical_text += (
            f"\nCustomer example: {case['customer_text']}\n"
            f"Historical AmazonHelp response: {case['brand_response']}\n"
        )

    prompt = f"""
{JUDGE_RUBRIC}

CUSTOMER MESSAGE:
{customer_message}

HISTORICAL EXAMPLES:
{historical_text}

AI-GENERATED REPLY:
{draft_reply}

Evaluate the AI reply.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    output = judge(
        messages,
        max_new_tokens=200,
        do_sample=False
    )[0]["generated_text"]

    # Extract the final JSON object
    matches = re.findall(
        r'\{.*?\}',
        output,
        flags=re.DOTALL
    )

    if not matches:
        return None

    try:
        result = json.loads(matches[-1])

        required = [
            "relevance",
            "historical_grounding",
            "actionability",
            "safety_accuracy",
            "overall"
        ]

        if not all(k in result for k in required):
            return None

        return result

    except Exception:
        return None

In [152]:
import json
import re

def judge_response(customer_message, historical_cases, draft_reply):

    historical_text = ""

    for _, case in historical_cases.iterrows():
        historical_text += (
            f"\nCustomer example: {case['customer_text']}\n"
            f"Historical AmazonHelp response: "
            f"{case['brand_response']}\n"
        )

    prompt = f"""
You are an evaluator for an AI customer-support agent.

Evaluate ONLY the quality of the AI-generated reply.

RUBRIC:

1. RELEVANCE
5 = directly addresses the customer's actual issue
4 = mostly addresses the issue
3 = partially relevant
2 = mostly misses the issue
1 = unrelated

2. HISTORICAL_GROUNDING
5 = clearly consistent with the supplied historical AmazonHelp resolutions
4 = mostly consistent
3 = plausible but weakly grounded
2 = little connection to historical evidence
1 = contradicts or ignores historical evidence

3. ACTIONABILITY
5 = gives a clear, useful next step
4 = useful but somewhat incomplete
3 = some guidance but vague
2 = very limited guidance
1 = no useful next step

4. SAFETY_ACCURACY
5 = no unsupported claims, unsafe advice, or unnecessary requests for sensitive information
4 = minor issue
3 = noticeable issue
2 = significant unsupported/problematic content
1 = clearly unsafe or misleading

IMPORTANT:
- Judge the reply against the CUSTOMER MESSAGE, not the predicted intent.
- Do not assume the AI's predicted intent is correct.
- Do not reward a reply merely because it is polite.
- If the reply addresses the wrong issue, give a low relevance score.
- If the reply invents unsupported Amazon policies or actions, reduce safety_accuracy.
- Use the historical examples as evidence of how AmazonHelp historically handled similar issues.

CUSTOMER MESSAGE:
{customer_message}

HISTORICAL EXAMPLES:
{historical_text}

AI-GENERATED REPLY:
{draft_reply}

Return ONLY valid JSON:

{{
  "relevance": 1,
  "historical_grounding": 1,
  "actionability": 1,
  "safety_accuracy": 1,
  "overall": 1
}}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    output = judge(
        messages,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )[0]["generated_text"]

    matches = re.findall(
        r'\{.*?\}',
        output,
        flags=re.DOTALL
    )

    if not matches:
        return None

    try:
        result = json.loads(matches[-1])

        required = [
            "relevance",
            "historical_grounding",
            "actionability",
            "safety_accuracy",
            "overall"
        ]

        if not all(k in result for k in required):
            return None

        # Validate scores
        for key in required:
            result[key] = int(result[key])
            if not 1 <= result[key] <= 5:
                return None

        return result

    except Exception:
        return None

In [155]:
# ============================================
# HUMAN EVALUATION SET — 5 CASES
# ============================================

human_sample = agent_eval.iloc[:5].copy()

human_sample = human_sample[
    [
        "customer_clean",
        "true_intent",
        "predicted_intent",
        "draft_reply"
    ]
].copy()

# Add empty columns for your manual ratings
human_sample["human_relevance"] = ""
human_sample["human_historical_grounding"] = ""
human_sample["human_actionability"] = ""
human_sample["human_safety_accuracy"] = ""
human_sample["human_overall"] = ""

human_sample.to_csv(
    "human_evaluation_5_cases.csv",
    index=False
)

display(human_sample)

,customer_clean,true_intent,predicted_intent,draft_reply,human_relevance,human_historical_grounding,human_actionability,human_safety_accuracy,human_overall
0,amazon prime package was said to be delivered ...,delivery_delayed,prime_subscription,I'm sorry you're having trouble with your Prim...,,,,,
1,"dear service provider ,my product not delivere...",delivery_not_received,delivery_delayed,I'm sorry your delivery is taking longer than ...,,,,,
2,Item ordered on 8th Oct not yet received. Deli...,delivery_delayed,delivery_delayed,I'm sorry your delivery is taking longer than ...,,,,,
3,Y a-t-il un moyen d'avoir une adresse de factu...,payment_or_billing,other,I'm sorry you're having trouble. Please provid...,,,,,
4,Queria saber se há previsão para a página de p...,product_information,payment_or_billing,I'm sorry you're having trouble with your paym...,,,,,


In [156]:
from google.colab import files

files.download("human_evaluation_5_cases.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>